> **This notebook is generated.** The canonical version of this project lives in its GitHub repo — [turbofan-maintenance-timing](https://github.com/Cena-S/turbofan-maintenance-timing) — and this file is produced from the repo's capstone notebook so the two can never disagree. If something here looks wrong, the repo is the thing to fix.


In [ ]:
# === GENERATED: inlined from src/ - do not edit, regenerate ===
#
# Kaggle cannot import this project's `src/` package, so every module the
# notebook uses is inlined below and registered in sys.modules. The cells that
# follow are byte-identical to the GitHub capstone.
#
# Canonical source: the repo. Regenerate with 05-communication/tools/make_kaggle_notebook.py

import sys, types

# Kaggle's writable dir stands in for the repo root, so modules that resolve paths
# from __file__ (PROJECT_ROOT = Path(__file__).parent.parent) still work.
_ROOT = '/kaggle/working'

_src = types.ModuleType('src'); _src.__path__ = [_ROOT + '/src']
_src.__file__ = _ROOT + '/src/__init__.py'
sys.modules['src'] = _src

# ---- src/acquisition.py ---------------------------------------------
_m = types.ModuleType('src.acquisition')
_m.__dict__['__name__'] = 'src.acquisition'
_m.__dict__['__file__'] = _ROOT + '/src/acquisition.py'
_m.__dict__['__package__'] = 'src'
_code_acquisition = r'''"""Download the NASA C-MAPSS turbofan degradation benchmark.

Acquisition is code, not a manual download: anyone cloning this repo can
reproduce the exact input by running this module. It records provenance (which
URL actually served, retrieval timestamp, SHA-256 of the archive and of every
extracted file, row and unit counts) so any number in the analysis traces back
to specific bytes.

What this data is
-----------------
NASA's Prognostics Center of Excellence ran the C-MAPSS engine simulator
(Commercial Modular Aero-Propulsion System Simulation) to failure on fleets of
simulated turbofan engines. Each engine starts with a different, unstated
amount of initial wear, runs under a control schedule, and degrades until it
no longer meets its operating specification. Every row is one engine on one
flight cycle: 3 operational settings and 21 sensor channels.

Four subsets vary two things — how many operating conditions the fleet flies
under, and how many independent fault modes can develop:

    FD001   100 train units   1 condition    1 fault  (HPC degradation)
    FD002   260 train units   6 conditions   1 fault  (HPC degradation)
    FD003   100 train units   1 condition    2 faults (HPC + fan)
    FD004   249 train units   6 conditions   2 faults (HPC + fan)

They are kept as four labelled subsets rather than merged blindly, because the
regime is a covariate the analysis needs: a six-condition fleet has sensor
readings that jump between operating points, and pooling it with a
single-condition fleet without a regime indicator would attribute that jump to
degradation.

Why this dataset, and what it does and does not stand for
--------------------------------------------------------
This project is about *when to pull a unit from service*, an industrial
maintenance-timing question. The honest position on the data:

* C-MAPSS is **simulated**, not measured. The degradation dynamics come from a
  physics model with injected noise, not from a real fleet. Nothing here should
  be read as a claim about real turbofans, and the README says so.
* It is used because it has the one property this analysis needs and almost no
  public maintenance dataset has: **complete run-to-failure trajectories for
  every unit**. That is what makes it possible to censor the data on purpose
  and then check the censoring-aware estimate against a known truth. A real
  fleet snapshot would not permit that check, which is the whole methodological
  spine of the project (see ``panel.py``).
* It is a **stated analogue**, not a substitute, for tooling-consumption and
  replenishment-timing work on real industrial equipment. It is not
  proprietary data from any employer and does not stand in for any.

Train vs test, and why the analysis uses the training trajectories
------------------------------------------------------------------
The benchmark ships each subset twice. ``train_FDxxx.txt`` runs every unit to
failure. ``test_FDxxx.txt`` truncates each unit some time before failure, with
the true remaining cycles supplied separately in ``RUL_FDxxx.txt`` — the
competition's held-out answer key.

The published leaderboard framing is "predict the RUL number in the answer
key". This project does not use that framing (``panel.py`` argues why at
length), so the training trajectories are what it models: they are the only
files with an observable failure event, which is what a survival model needs.
The test files and RUL keys are still downloaded and checksummed — they are
used once, as an external check that the induced-censoring construction
produces a fleet whose truncation pattern resembles the benchmark's own.

Source
------
Saxena, A. and Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data
Set*. NASA Prognostics Data Repository, NASA Ames Research Center, Moffett
Field, CA. https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/

Licence: work of the U.S. Government, public domain. Redistribution is
permitted, but the data is not committed to this repository — downloading at
runtime is what guarantees the provenance record describes the file actually
analysed.

Usage
-----
    python src/acquisition.py
"""

from __future__ import annotations

import hashlib
import io
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTRACT_DIR = RAW_DIR / "CMAPSSData"
ARCHIVE_PATH = RAW_DIR / "CMAPSSData.zip"
PROVENANCE_PATH = RAW_DIR / "provenance.json"
TRAIN_PARQUET = RAW_DIR / "cmapss_train.parquet"
TEST_PARQUET = RAW_DIR / "cmapss_test.parquet"
RUL_PARQUET = RAW_DIR / "cmapss_test_rul.parquet"

SOURCE_CITATION = (
    "Saxena, A. and Goebel, K. (2008). Turbofan Engine Degradation Simulation "
    "Data Set. NASA Prognostics Data Repository, NASA Ames Research Center, "
    "Moffett Field, CA."
)
SOURCE_PAGE = (
    "https://www.nasa.gov/intelligent-systems-division/"
    "discovery-and-systems-health/pcoe/pcoe-data-set-repository/"
)
SOURCE_LICENCE = "U.S. Government work — public domain"

# NASA has moved this file more than once: the original ti.arc.nasa.gov host was
# retired, and data.nasa.gov has reorganised since. Rather than pin one URL and
# break, the module tries known-good mirrors in order and records which one
# actually served. All three are expected to deliver byte-identical archives;
# the checksum in the provenance record is what proves it for a given run.
DOWNLOAD_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://phm-datasets.s3.amazonaws.com/NASA/"
    "6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]

# The four subsets, with the design NASA published for each. Carried in code
# rather than looked up later because the regime is a modelling covariate: the
# number of operating conditions decides whether sensor jumps are degradation
# or just a change of operating point.
SUBSETS = {
    "FD001": {"conditions": 1, "faults": 1, "fault_modes": "HPC degradation"},
    "FD002": {"conditions": 6, "faults": 1, "fault_modes": "HPC degradation"},
    "FD003": {"conditions": 1, "faults": 2, "fault_modes": "HPC degradation, fan degradation"},
    "FD004": {"conditions": 6, "faults": 2, "fault_modes": "HPC degradation, fan degradation"},
}

# Unit counts published with the benchmark. Asserted after parsing, so a
# truncated download or a repackaged mirror that quietly dropped units fails
# loudly instead of shrinking the fleet by a few engines.
EXPECTED_TRAIN_UNITS = {"FD001": 100, "FD002": 260, "FD003": 100, "FD004": 249}
EXPECTED_TEST_UNITS = {"FD001": 100, "FD002": 259, "FD003": 100, "FD004": 248}

SENSOR_COLUMNS = [f"sensor_{i}" for i in range(1, 22)]
SETTING_COLUMNS = ["setting_1", "setting_2", "setting_3"]
DATA_COLUMNS = ["unit", "cycle", *SETTING_COLUMNS, *SENSOR_COLUMNS]

CHUNK_BYTES = 1 << 16
TIMEOUT = 300

USER_AGENT = (
    "turbofan-maintenance-timing/1.0 (portfolio data analysis project; "
    "contact via GitHub Cena-S)"
)


def make_session() -> requests.Session:
    """A session that backs off rather than hammering a mirror that is failing."""
    retry = Retry(
        total=4,
        backoff_factor=2.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def _sha256(path: Path) -> str:
    """Checksum a file without reading it entirely into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_archive(force: bool = False) -> tuple[Path, str]:
    """Fetch the archive from the first mirror that serves it.

    Returns the path and the URL that worked. Recording *which* mirror served
    matters: if two runs produce different checksums, the provenance record is
    what identifies the mirror that changed rather than leaving it a mystery.
    """
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    if ARCHIVE_PATH.exists() and not force:
        recorded = _recorded_source_url()
        print(f"Archive already present: {ARCHIVE_PATH.name}")
        return ARCHIVE_PATH, recorded or "cached (source URL not recorded)"

    session = make_session()
    failures: list[str] = []

    for url in DOWNLOAD_URLS:
        try:
            print(f"Trying {url}")
            response = session.get(url, stream=True, timeout=TIMEOUT)
            response.raise_for_status()
            with ARCHIVE_PATH.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=CHUNK_BYTES):
                    handle.write(chunk)
            # A mirror that returns an HTML error page with a 200 would
            # otherwise be caught much later, as an unreadable data file.
            if not zipfile.is_zipfile(ARCHIVE_PATH):
                ARCHIVE_PATH.unlink(missing_ok=True)
                raise RuntimeError("served content is not a zip archive")
            print(f"  served {ARCHIVE_PATH.stat().st_size / 1e6:.1f} MB")
            return ARCHIVE_PATH, url
        except Exception as error:  # noqa: BLE001 - the point is to try the next mirror
            failures.append(f"{url}: {error}")
            print(f"  failed: {error}")

    raise RuntimeError(
        "Every known mirror failed. Tried:\n  " + "\n  ".join(failures)
        + "\n\nFallback: download the C-MAPSS archive manually from\n  "
        + SOURCE_PAGE
        + f"\nand place it at {ARCHIVE_PATH}, then re-run this module."
    )


def _recorded_source_url() -> str | None:
    """The URL a previous run recorded, if there is one."""
    if not PROVENANCE_PATH.exists():
        return None
    try:
        return json.loads(PROVENANCE_PATH.read_text(encoding="utf-8"))["source_url"]
    except (json.JSONDecodeError, KeyError):
        return None


def extract_archive(archive: Path) -> dict[str, dict]:
    """Unpack the data files, following one level of nesting if present.

    The mirrors differ in packaging: data.nasa.gov serves ``CMAPSSData.zip``
    directly, while the PHM S3 bucket serves an outer zip with that archive
    inside it. Handling the nested case is four lines and removes an entire
    class of "worked on my machine" failure.

    Only the ``*_FDxxx.txt`` data files are extracted; directory structure in
    the archive is flattened so downstream code has one place to look.
    """
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    extracted: dict[str, dict] = {}

    def _extract_from(source: zipfile.ZipFile, depth: int = 0) -> None:
        for info in source.infolist():
            if info.is_dir():
                continue
            name = Path(info.filename).name
            if name.lower().endswith(".zip") and depth == 0:
                with source.open(info) as nested_handle:
                    payload = io.BytesIO(nested_handle.read())
                with zipfile.ZipFile(payload) as nested:
                    _extract_from(nested, depth + 1)
                continue
            if not name.endswith(".txt"):
                continue
            if not any(name.startswith(prefix) for prefix in ("train_", "test_", "RUL_")):
                continue
            target = EXTRACT_DIR / name
            with source.open(info) as handle, target.open("wb") as out:
                out.write(handle.read())
            extracted[name] = {
                "sha256": _sha256(target),
                "bytes": target.stat().st_size,
            }

    with zipfile.ZipFile(archive) as archive_zip:
        _extract_from(archive_zip)

    missing = [
        f"{kind}_{subset}.txt"
        for subset in SUBSETS
        for kind in ("train", "test", "RUL")
    ]
    absent = [name for name in missing if name not in extracted]
    if absent:
        raise RuntimeError(
            f"Archive is missing expected data files: {absent}. "
            "The mirror may have repackaged the dataset."
        )

    print(f"Extracted {len(extracted)} data files to {EXTRACT_DIR.name}/")
    return extracted


def read_trajectory_file(path: Path, subset: str) -> pd.DataFrame:
    """Read one ``train_`` or ``test_`` file into the canonical 26-column frame.

    The files are space-delimited with *trailing* whitespace on every line, so
    a naive read produces two phantom all-NaN columns at the end. Using
    ``sep=r"\\s+"`` with explicit names and dropping the surplus is more
    honest than silently accepting 28 columns and indexing positionally.
    """
    frame = pd.read_csv(path, sep=r"\s+", header=None, engine="python")

    if frame.shape[1] < len(DATA_COLUMNS):
        raise RuntimeError(
            f"{path.name}: expected at least {len(DATA_COLUMNS)} columns, "
            f"found {frame.shape[1]}."
        )

    frame = frame.iloc[:, : len(DATA_COLUMNS)]
    frame.columns = DATA_COLUMNS
    frame["unit"] = frame["unit"].astype("int32")
    frame["cycle"] = frame["cycle"].astype("int32")
    frame["subset"] = subset

    # Cycles must be a complete 1..n run per unit. A gap would silently change
    # every duration in the analysis, so it is checked rather than assumed.
    per_unit = frame.groupby("unit")["cycle"]
    if not (per_unit.min() == 1).all():
        raise RuntimeError(f"{path.name}: some units do not start at cycle 1.")
    if not (per_unit.max() == per_unit.count()).all():
        raise RuntimeError(f"{path.name}: some units have gaps in their cycle index.")

    return frame


def read_rul_file(path: Path, subset: str) -> pd.DataFrame:
    """Read the answer key: true remaining cycles for each truncated test unit."""
    values = pd.read_csv(path, sep=r"\s+", header=None, engine="python").iloc[:, 0]
    return pd.DataFrame({
        "subset": subset,
        "unit": range(1, len(values) + 1),
        "true_rul": values.astype("int32").to_numpy(),
    })


def _assemble(kind: str, expected_units: dict[str, int]) -> tuple[pd.DataFrame, list[dict]]:
    """Read all four subsets of one kind and check the published unit counts."""
    frames: list[pd.DataFrame] = []
    records: list[dict] = []

    for subset in SUBSETS:
        frame = read_trajectory_file(EXTRACT_DIR / f"{kind}_{subset}.txt", subset)
        units = frame["unit"].nunique()

        if units != expected_units[subset]:
            raise RuntimeError(
                f"{kind}_{subset}.txt: expected {expected_units[subset]} units, "
                f"found {units}. The mirror may not be the published dataset."
            )

        frames.append(frame)
        records.append({
            "subset": subset,
            "kind": kind,
            "units": int(units),
            "rows": int(frame.shape[0]),
            "max_cycles": int(frame["cycle"].max()),
            "median_unit_length": float(frame.groupby("unit")["cycle"].max().median()),
            **SUBSETS[subset],
        })
        print(
            f"  {kind:>5} {subset}  {units:>3} units  {frame.shape[0]:>6,} rows  "
            f"median life {records[-1]['median_unit_length']:.0f} cycles"
        )

    return pd.concat(frames, ignore_index=True), records


def main(force: bool = False) -> pd.DataFrame:
    archive, served_url = download_archive(force=force)
    archive_sha = _sha256(archive)
    extracted = extract_archive(archive)

    print("\nParsing subsets:")
    train, train_records = _assemble("train", EXPECTED_TRAIN_UNITS)
    test, test_records = _assemble("test", EXPECTED_TEST_UNITS)

    ruls = pd.concat(
        [read_rul_file(EXTRACT_DIR / f"RUL_{subset}.txt", subset) for subset in SUBSETS],
        ignore_index=True,
    )
    for subset in SUBSETS:
        n_keys = int((ruls["subset"] == subset).sum())
        if n_keys != EXPECTED_TEST_UNITS[subset]:
            raise RuntimeError(
                f"RUL_{subset}.txt: {n_keys} answers for "
                f"{EXPECTED_TEST_UNITS[subset]} test units."
            )

    train.to_parquet(TRAIN_PARQUET, index=False)
    test.to_parquet(TEST_PARQUET, index=False)
    ruls.to_parquet(RUL_PARQUET, index=False)

    record = {
        "citation": SOURCE_CITATION,
        "source_page": SOURCE_PAGE,
        "source_url": served_url,
        "candidate_urls": DOWNLOAD_URLS,
        "licence": SOURCE_LICENCE,
        "data_caveat": (
            "C-MAPSS is simulator output, not measurements from a real fleet. "
            "Used here because complete run-to-failure trajectories are what "
            "make the censoring check possible; no claim about real turbofans "
            "is made or implied."
        ),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "archive": {
            "filename": archive.name,
            "sha256": archive_sha,
            "bytes": archive.stat().st_size,
        },
        "files": extracted,
        "train": train_records,
        "test": test_records,
        "totals": {
            "train_units": int(train.groupby(["subset", "unit"]).ngroups),
            "train_rows": int(train.shape[0]),
            "test_units": int(test.groupby(["subset", "unit"]).ngroups),
            "test_rows": int(test.shape[0]),
        },
    }
    PROVENANCE_PATH.write_text(json.dumps(record, indent=2), encoding="utf-8")

    totals = record["totals"]
    print(
        f"\n{totals['train_units']} training units "
        f"({totals['train_rows']:,} unit-cycles) -> {TRAIN_PARQUET.name}"
    )
    print(
        f"{totals['test_units']} test units "
        f"({totals['test_rows']:,} unit-cycles) -> {TEST_PARQUET.name}"
    )
    print(f"Provenance -> {PROVENANCE_PATH.name}  (served by {served_url})")
    return train


if __name__ == "__main__":
    main()
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.acquisition'] = _m
setattr(_src, 'acquisition', _m)
exec(compile(_code_acquisition, 'src/acquisition.py', 'exec'), _m.__dict__)

# ---- src/panel.py ---------------------------------------------------
_m = types.ModuleType('src.panel')
_m.__dict__['__name__'] = 'src.panel'
_m.__dict__['__file__'] = _ROOT + '/src/panel.py'
_m.__dict__['__package__'] = 'src'
_code_panel = r'''"""Turn run-to-failure trajectories into a censored fleet, then into a risk set.

This module carries the methodological argument of the whole project. It does
two things: it *removes* information on purpose, and it reshapes what is left
into the counting-process format survival models need.

Why remove information on purpose
---------------------------------
Almost every published C-MAPSS notebook does the same thing: regress remaining
useful life on the sensor readings, report RMSE against the benchmark's answer
key. That framing quietly assumes you can see every engine's failure. An
operator never can. At any moment the fleet is mostly units that have *not*
failed yet — and those are exactly the units a maintenance decision is about.
A method that needs a completed lifetime to learn from cannot learn from the
fleet you actually have.

So this module builds the fleet an operator would actually see. Each unit is
observed from its first cycle up to a snapshot horizon drawn independently of
how long it was going to last. If it failed before the snapshot, the failure
is observed. If not, all that is known is that it survived that long: the
observation is **right-censored**.

That costs information, and the cost is the point. Because the untruncated
trajectories still exist in ``data/raw``, the analysis can do something a real
fleet snapshot never permits — compare three things side by side:

1. the **known truth**, estimated on the full run-to-failure data;
2. the **complete-case** estimate, which drops censored units (what "regress on
   RUL" effectively does, because a unit with no observed failure has no
   target);
3. the **censoring-aware** estimate, which keeps them.

The complete-case estimate is not merely less precise. It is *biased*, and
biased in a direction that is easy to state: a unit is censored precisely when
its lifetime exceeds its horizon, so the units that survive the filter are
disproportionately the short-lived ones. Dropping censored units therefore
throws away the long lives specifically, and the fleet looks more fragile than
it is. ``evaluate.py`` measures how much.

The censoring mechanism, stated so it can be argued with
--------------------------------------------------------
Unit *i* has a true failure cycle ``T_i`` (known from the raw data, hidden from
every model). It is assigned a snapshot horizon ``C_i`` drawn uniformly from
``{1, …, U}``, independently of ``T_i``. The unit is then observed for
``Y_i = min(T_i, C_i)`` cycles, with event indicator ``δ_i = 1{T_i ≤ C_i}``.

Three properties this buys, each of which matters:

* **Independent censoring holds by construction.** ``C_i`` is drawn from a
  distribution that does not see ``T_i`` or any sensor reading. Kaplan-Meier
  and partial likelihood both require this, and here it is true by design
  rather than hoped for. (Informative censoring — units pulled *because* they
  looked unwell — is a real phenomenon in maintenance data and would break
  these estimators. It is out of scope and named in the README's limitations.)
* **It is administrative, not left-truncating.** The unit is watched from cycle
  1, so every unit's full history up to the snapshot is available. There is no
  delayed entry to handle.
* **It is reproducible.** ``U`` is solved for deterministically to hit a target
  censored fraction, and horizons are drawn from a seeded generator. Two runs
  produce the same fleet.

``U`` is *calibrated*, not guessed: given the lifetimes, the censored fraction
is a monotone decreasing function of ``U``, so a bisection finds the ``U`` that
hits the target rate. The default target is 40%, and ``run_analysis.py`` sweeps
it (20/40/60%) so the conclusions are not an artefact of one setting.

Operating regime, and why the sensors are normalised inside it
--------------------------------------------------------------
FD002 and FD004 fly six operating conditions; FD001 and FD003 fly one. In the
six-condition subsets the raw sensor variance is dominated by *which condition
the engine is in*, not by how worn it is — sensor 7's raw standard deviation is
0.9 in FD001 and 146 in FD002, and that hundredfold difference is altitude and
throttle, not degradation.

Feeding raw sensors to a model across subsets would therefore let it learn the
flight profile and call it wear. Each sensor is standardised **within operating
regime** instead, so a reading means "unusual for this operating point".

The six regimes are recovered by rounding the three operational settings, not
by clustering. The settings fall into six tight, well-separated groups (checked
in notebook 02), so rounding recovers them exactly and — unlike k-means — is
deterministic, needs no seed, and can be read off in one line. Adding ``+ 0.0``
before formatting is not cosmetic: FD001 stores its setting as ``-0.0``, which
would otherwise label the same physical condition differently from FD002's
``0.0`` and produce a phantom seventh regime.

Sensor screening
----------------
Four sensors (1, 5, 18, 19) are exactly constant within every regime, and two
more (10, 16) are near-constant. Constant columns carry no information and make
the design matrix singular. They are dropped by a variance rule applied to the
data rather than by a hard-coded list, so the screen is reproducible on any
subset selection — and the dropped set is reported rather than silently
removed.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

RANDOM_SEED = 20260801

# Target share of units still running at the snapshot. 40% is a plausible
# steady-state fleet: enough failures to estimate from, enough survivors that
# ignoring them is clearly wrong. Swept in run_analysis.py.
DEFAULT_CENSOR_RATE = 0.40

# A sensor whose within-regime standard deviation is below this everywhere is
# treated as constant. Set well below the smallest genuinely-varying sensor
# (sensor 16, ~6e-4) and well above exact zero, so the rule separates the two
# groups cleanly instead of splitting a continuum.
MIN_WITHIN_REGIME_STD = 1e-6

# Trailing window for the smoothed sensor level. C-MAPSS sensor noise is large
# relative to the per-cycle degradation increment, so a single reading is a
# poor state estimate. Five cycles is short enough to stay responsive and long
# enough to halve the noise. Strictly trailing: a feature at cycle t may never
# use a reading from cycle t+1, or the model would be predicting failure from
# evidence collected after the decision it is meant to inform.
ROLLING_WINDOW = 5

SENSOR_COLUMNS = [f"sensor_{i}" for i in range(1, 22)]
SETTING_COLUMNS = ["setting_1", "setting_2", "setting_3"]

HOLDOUT_FRACTION = 0.30


def assign_regime(frame: pd.DataFrame) -> pd.Series:
    """Label each row with its operating regime.

    Rounding rather than clustering: the settings form six tight groups, so
    this recovers them exactly and deterministically. ``+ 0.0`` normalises
    negative zero, which otherwise splits one physical condition into two.
    """
    s1 = (frame["setting_1"].round(0) + 0.0).astype(int)
    s2 = (frame["setting_2"].round(2) + 0.0).round(2)
    s3 = (frame["setting_3"].round(0) + 0.0).astype(int)
    return pd.Series(
        [f"{a}|{b:.2f}|{c}" for a, b, c in zip(s1, s2, s3)],
        index=frame.index,
        name="regime",
        dtype="string",
    )


def screen_sensors(frame: pd.DataFrame,
                   min_std: float = MIN_WITHIN_REGIME_STD) -> tuple[list[str], list[str]]:
    """Split sensors into those that vary within regime and those that do not.

    Returns ``(kept, dropped)``. Applied to the data rather than hard-coded, so
    the screen stays correct if the subset selection changes.
    """
    within_std = frame.groupby("regime", observed=True)[SENSOR_COLUMNS].std()
    varies = (within_std.fillna(0.0) > min_std).any(axis=0)
    kept = [column for column in SENSOR_COLUMNS if varies.get(column, False)]
    dropped = [column for column in SENSOR_COLUMNS if column not in kept]
    return kept, dropped


def normalise_within_regime(frame: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    """Standardise each sensor against its own operating regime.

    A value of +2 then means "two standard deviations high *for this operating
    point*", which is comparable across a six-condition fleet and a
    single-condition one. Without this, a model pooling the subsets would learn
    the flight profile and mistake it for wear.
    """
    grouped = frame.groupby("regime", observed=True)[sensors]
    centred = frame[sensors] - grouped.transform("mean")
    spread = grouped.transform("std").replace(0.0, np.nan)
    return (centred / spread).fillna(0.0).add_suffix("_z")


def add_dynamics(frame: pd.DataFrame, sensors: list[str],
                 window: int = ROLLING_WINDOW) -> pd.DataFrame:
    """Add the trailing state estimate and the drift away from the unit's own start.

    Two derived features per sensor, both strictly backward-looking:

    * ``_roll`` — trailing mean over ``window`` cycles. The state estimate the
      raw reading is too noisy to be.
    * ``_drift`` — that trailing mean minus the unit's own first-``window``
      average. Engines enter service with different, unstated initial wear, so
      the *level* of a sensor confounds "born worn" with "worn out". The change
      since the unit's own baseline separates them, and it is the quantity a
      maintenance engineer actually watches.

    Both are computed per unit and use only cycles at or before the current
    one, so a feature never sees the future.
    """
    z_columns = [f"{sensor}_z" for sensor in sensors]
    grouped = frame.groupby(["subset", "unit"], observed=True)

    rolled = (
        grouped[z_columns]
        .transform(lambda column: column.rolling(window, min_periods=1).mean())
        .add_suffix("_roll")
    )
    rolled.columns = [column.replace("_z_roll", "_roll") for column in rolled.columns]

    baseline = (
        grouped[z_columns]
        .transform(lambda column: column.head(window).mean())
    )
    drift = (rolled.to_numpy() - baseline.to_numpy())
    drift = pd.DataFrame(
        drift,
        index=frame.index,
        columns=[column.replace("_z", "_drift") for column in z_columns],
    )
    return pd.concat([rolled, drift], axis=1)


def unit_lifetimes(frame: pd.DataFrame) -> pd.DataFrame:
    """The true failure cycle of every unit — the ground truth the models never see."""
    lifetimes = (
        frame.groupby(["subset", "unit"], observed=True)["cycle"]
        .max()
        .reset_index()
        .rename(columns={"cycle": "true_lifetime"})
    )
    return lifetimes.sort_values(["subset", "unit"]).reset_index(drop=True)


def calibrate_horizon(lifetimes: np.ndarray, target_rate: float) -> int:
    """Find the horizon ceiling ``U`` that produces the target censored fraction.

    With ``C ~ Uniform{1..U}`` drawn independently of ``T``, a unit is censored
    exactly when ``C < T``, so

        P(censored) = mean_i  min(T_i - 1, U) / U

    which decreases monotonically in ``U``. Bisection therefore finds the
    unique ``U`` hitting the target, and the censored fraction becomes a
    *chosen* property of the design rather than whatever happened to fall out
    of an arbitrary constant.
    """
    if not 0.0 < target_rate < 1.0:
        raise ValueError(f"target_rate must be strictly between 0 and 1, got {target_rate}")

    def expected_rate(ceiling: int) -> float:
        return float(np.mean(np.minimum(lifetimes - 1, ceiling) / ceiling))

    low, high = 1, int(lifetimes.max())
    # Grow the ceiling until the achievable rate brackets the target. A very
    # low target needs a horizon well beyond the longest lifetime.
    while expected_rate(high) > target_rate:
        high *= 2
        if high > 10_000 * int(lifetimes.max()):
            raise RuntimeError(f"Could not reach a censored fraction of {target_rate}.")

    while low < high:
        middle = (low + high) // 2
        if expected_rate(middle) > target_rate:
            low = middle + 1
        else:
            high = middle
    return int(low)


def induce_censoring(lifetimes: pd.DataFrame,
                     target_rate: float = DEFAULT_CENSOR_RATE,
                     seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Draw a snapshot horizon per unit and record what would have been observed.

    Adds four columns to the lifetimes table:

    ``horizon``        the snapshot cycle ``C_i``, drawn independently of ``T_i``
    ``observed_time``  ``Y_i = min(T_i, C_i)`` — how long the unit was watched
    ``event``          ``1`` if the failure was seen, ``0`` if still running
    ``censored``       the complement, kept for readability at call sites

    ``true_lifetime`` stays in the frame deliberately. It is the ground truth
    used to score the three estimates against each other, and every function
    that fits a model takes the censored columns only — never this one.
    """
    result = lifetimes.copy()
    ceiling = calibrate_horizon(result["true_lifetime"].to_numpy(), target_rate)

    generator = np.random.default_rng(seed)
    result["horizon"] = generator.integers(1, ceiling + 1, size=len(result))
    result["observed_time"] = np.minimum(result["true_lifetime"], result["horizon"])
    result["event"] = (result["true_lifetime"] <= result["horizon"]).astype(int)
    result["censored"] = 1 - result["event"]
    result.attrs["horizon_ceiling"] = ceiling
    result.attrs["target_censor_rate"] = target_rate
    result.attrs["realised_censor_rate"] = float(result["censored"].mean())
    result.attrs["seed"] = seed
    return result


def split_units(lifetimes: pd.DataFrame,
                holdout_fraction: float = HOLDOUT_FRACTION,
                seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Assign each unit to fit or holdout, stratified by subset.

    The split is by **unit**, never by row: a unit contributes hundreds of
    correlated cycle observations, and splitting rows would put a unit's early
    cycles in the fit set and its late cycles in the holdout, letting the model
    memorise the engine it is about to be scored on.

    Stratifying by subset keeps the four operating-condition/fault-mode designs
    represented in both halves, so holdout performance is not dominated by
    whichever regime happened to land there.
    """
    generator = np.random.default_rng(seed + 1)
    result = lifetimes.copy()
    result["split"] = "fit"

    for subset, group in result.groupby("subset", observed=True):
        n_holdout = int(round(len(group) * holdout_fraction))
        chosen = generator.choice(group.index.to_numpy(), size=n_holdout, replace=False)
        result.loc[chosen, "split"] = "holdout"

    return result


def build_risk_set(trajectories: pd.DataFrame, fleet: pd.DataFrame) -> pd.DataFrame:
    """Assemble the unit-cycle risk set the models are fitted on.

    One row per unit per cycle it was *at risk* — that is, cycles 1 through
    ``observed_time``. Cycles after the snapshot are dropped because they were
    never seen; that is what censoring means, and keeping them would be the
    leak the whole exercise is designed to avoid.

    The outcome ``failed`` is 1 only on a unit's final row *and* only if the
    failure was observed. For a censored unit every row is 0: it never failed
    within the window, which is all the data says.

    The baseline-hazard banding is deliberately *not* built here. Its cut
    points must be derived from the fit set alone, or they leak the holdout's
    failure times into the design, so they belong with the model —
    ``survival.derive_band_edges``.
    """
    fleet_columns = ["subset", "unit", "observed_time", "event", "true_lifetime", "split"]
    merged = trajectories.merge(
        fleet.loc[:, [column for column in fleet_columns if column in fleet.columns]],
        on=["subset", "unit"],
        how="inner",
        validate="many_to_one",
    )

    at_risk = merged.loc[merged["cycle"] <= merged["observed_time"]].copy()

    is_last_cycle = at_risk["cycle"] == at_risk["observed_time"]
    at_risk["failed"] = (is_last_cycle & (at_risk["event"] == 1)).astype(int)

    # Counting-process interval for this cycle: (entry, exit].
    at_risk["entry"] = at_risk["cycle"] - 1
    at_risk["exit"] = at_risk["cycle"]

    # A stable per-unit key. The raw unit numbers restart at 1 in each subset,
    # so clustering or grouping on `unit` alone would silently merge four
    # different engines into one.
    at_risk["unit_id"] = at_risk["subset"].astype(str) + "_" + at_risk["unit"].astype(str)

    return at_risk.sort_values(["subset", "unit", "cycle"]).reset_index(drop=True)


def build_evaluation_panel(trajectories: pd.DataFrame, fleet: pd.DataFrame) -> pd.DataFrame:
    """Full, untruncated trajectories — for scoring policies, never for fitting.

    A maintenance policy has to be scored on what *would have happened*: if the
    alarm fires at cycle 190 and the engine would have run to 240, the policy
    bought 50 cycles of avoided failure and gave up 50 cycles of service. That
    counterfactual needs the whole trajectory, including the part after the
    snapshot that the models are forbidden to see.

    Hence two panels, and the separation between them is load-bearing:

    * ``risk_set`` — truncated at the snapshot. Everything that fits a model
      uses this and only this.
    * ``evaluation_panel`` — the complete run to failure. Used only to simulate
      what a policy would have cost.

    Being able to do this at all is the reason a simulator benchmark is worth
    using. On a real fleet the counterfactual is unobservable and a policy can
    only be argued for, never measured. The price is that the *evaluation* is
    as realistic as C-MAPSS is, which the README states plainly.
    """
    fleet_columns = ["subset", "unit", "observed_time", "event", "true_lifetime", "split"]
    panel = trajectories.merge(
        fleet.loc[:, [column for column in fleet_columns if column in fleet.columns]],
        on=["subset", "unit"],
        how="inner",
        validate="many_to_one",
    )
    panel["unit_id"] = panel["subset"].astype(str) + "_" + panel["unit"].astype(str)
    return panel.sort_values(["subset", "unit", "cycle"]).reset_index(drop=True)


def prepare(trajectories: pd.DataFrame,
            target_rate: float = DEFAULT_CENSOR_RATE,
            seed: int = RANDOM_SEED) -> dict:
    """Run the whole construction: regimes, features, censoring, split, risk set.

    Returns a dict with the risk set, the fleet table, the sensors kept and
    dropped, and the censoring design — everything the notebooks and
    ``run_analysis.py`` need, assembled in one place so the capstone notebook
    and the script can never build it differently.
    """
    frame = trajectories.copy()
    frame["regime"] = assign_regime(frame)

    kept, dropped = screen_sensors(frame)
    normalised = normalise_within_regime(frame, kept)
    frame = pd.concat([frame, normalised], axis=1)
    frame = pd.concat([frame, add_dynamics(frame, kept)], axis=1)

    lifetimes = unit_lifetimes(trajectories)
    fleet = induce_censoring(lifetimes, target_rate=target_rate, seed=seed)
    fleet = split_units(fleet, seed=seed)
    fleet.attrs = {**lifetimes.attrs, **fleet.attrs}

    risk_set = build_risk_set(frame, fleet)
    evaluation_panel = build_evaluation_panel(frame, fleet)

    return {
        "risk_set": risk_set,
        "evaluation_panel": evaluation_panel,
        "fleet": fleet,
        "sensors_kept": kept,
        "sensors_dropped": dropped,
        # Drift, not level, is the model's feature set. The argument is in
        # `add_dynamics`: engines enter service with different unstated initial
        # wear, so a sensor's level confounds "born worn" with "worn out".
        # The claim is not left as an argument — `evaluate.compare_feature_sets`
        # fits all three and reports that levels fit better in sample and
        # discriminate worse out of it, which is what overfitting looks like.
        "feature_columns": [f"{sensor}_drift" for sensor in kept],
        "level_columns": [f"{sensor}_roll" for sensor in kept],
        "n_regimes": int(frame["regime"].nunique()),
        "censoring": {
            "target_rate": target_rate,
            "realised_rate": float(fleet["censored"].mean()),
            "horizon_ceiling": int(fleet.attrs["horizon_ceiling"]),
            "seed": seed,
        },
    }
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.panel'] = _m
setattr(_src, 'panel', _m)
exec(compile(_code_panel, 'src/panel.py', 'exec'), _m.__dict__)

# ---- src/survival.py ------------------------------------------------
_m = types.ModuleType('src.survival')
_m.__dict__['__name__'] = 'src.survival'
_m.__dict__['__file__'] = _ROOT + '/src/survival.py'
_m.__dict__['__package__'] = 'src'
_code_survival = r'''"""Four views of when an engine fails, in increasing order of what they assume.

The models are deliberately laid out as a ladder, because the point of the
project is the *decision*, and each rung exists to justify the next one:

1. **Kaplan-Meier + log-rank** — no covariates, no functional form. Establishes
   that the four subsets have genuinely different survival before any model is
   allowed to assume so.
2. **Cox proportional hazards with time-varying covariates** — the reflex
   choice for duration data. Fitted, and then its central assumption is
   *tested* rather than inherited.
3. **Discrete-time logistic hazard** — the workhorse. Produces the calibrated
   per-cycle failure probability the maintenance decision actually needs.
4. **Random Survival Forest** — a comparator whose only job is to bound how
   much discrimination the interpretable model gives up.

Why the discrete-time model is the workhorse and not Cox
--------------------------------------------------------
Three reasons, in order of weight:

* **The decision needs an absolute probability, not a ranking.** Cox's partial
  likelihood deliberately factors out the baseline hazard: it estimates
  *relative* risk and is silent about the absolute level unless a baseline is
  bolted back on afterwards. "Pull the unit when its 30-cycle failure risk
  exceeds p" is a statement about an absolute probability, so a model that
  estimates one directly is the honest fit to the question.
* **Time is discrete here.** An engine is inspected once per flight cycle and
  either fails or does not. That is a Bernoulli trial per cycle at risk, which
  is exactly what a logistic regression on the risk set estimates. Cox assumes
  continuous time in which exact ties cannot occur; with integer cycles, ties
  are not a nuisance to approximate away, they are the data.
* **Duration dependence is the question.** Whether the hazard accelerates with
  age is precisely what a maintenance policy turns on. Entering cycle as a set
  of band dummies lets the hazard take any shape; a linear term would impose
  the answer.

Cox is still fitted, because "I used the simpler model" is only credible if
the standard one was tried and its assumption checked. The PH test in
``test_proportional_hazards`` is that check.

Standard errors
---------------
Clustered by unit throughout. One engine contributes hundreds of cycle
observations that are anything but independent — a unit degrading fast does so
on every one of its rows. Treating them as independent would shrink the
standard errors by roughly the square root of the average unit length and make
every coefficient look far more precise than it is.

Predicting forward, and the assumption that makes it operational
----------------------------------------------------------------
The decision layer needs P(fail within the next *h* cycles | alive now). From a
discrete-time hazard that is

    risk_h(t) = 1 - Π_{s=t+1..t+h} (1 - λ(s, x))

which requires covariates at future cycles that have not happened yet. They are
held at their current value. This is not a convenience: at the moment of the
decision, today's readings are all the operator has, so a model that needed
tomorrow's would be unusable. The cost is that a unit degrading unusually fast
has its future risk understated, which makes the reported policy gain
conservative rather than flattering.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import statsmodels.api as sm
from lifelines import CoxTimeVaryingFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

RANDOM_SEED = 20260801

# Cycles ahead the maintenance alarm looks. Chosen to be an operationally
# meaningful lead time — long enough to schedule a shop visit, short enough
# that the frozen-covariate assumption above is not doing heavy lifting.
# Swept in run_analysis.py so the policy conclusion is not tied to one horizon.
DEFAULT_HORIZON = 30

# Landmark cycles at which the models are scored. Landmarking is what makes a
# time-varying-covariate model comparable to a plain survival model: fix a
# cycle, keep the units still at risk then, use their readings at that instant,
# and score how well the predicted risk orders what happens next. Without it,
# "the model's C-index" is ambiguous about which covariate values it used.
LANDMARK_CYCLES = (50, 100, 150)

# Bands the baseline hazard is estimated in. Ten gives roughly 30 observed
# failures per band in the fit set — enough to estimate a level, few enough
# that the shape is not read out of noise.
N_HAZARD_BANDS = 10

# The band left out of the design as the reference category. Band 1 is the
# youngest band in which failures occur; band 0 (younger than any observed
# failure) is empty once the risk window is applied, so it cannot serve.
BAND_REFERENCE = 1


def kaplan_meier(fleet: pd.DataFrame, group_column: str = "subset") -> pd.DataFrame:
    """Survival curves by group, handling censoring correctly.

    A censored unit contributes everything it knows — that it lasted at least
    this long — without being counted as having failed. That is the whole
    reason to use Kaplan-Meier rather than averaging observed lifetimes, and
    the gap between the two is measured in ``evaluate.py``.
    """
    curves: list[pd.DataFrame] = []
    fitter = KaplanMeierFitter()

    for group, subset in fleet.groupby(group_column, observed=True):
        fitter.fit(
            subset["observed_time"],
            event_observed=subset["event"].astype(bool),
            label=str(group),
        )
        frame = fitter.survival_function_.reset_index()
        frame.columns = ["cycle", "survival"]
        confidence = fitter.confidence_interval_.reset_index(drop=True)
        frame["lower"] = confidence.iloc[:, 0].to_numpy()
        frame["upper"] = confidence.iloc[:, 1].to_numpy()
        frame["group"] = str(group)
        frame["units"] = len(subset)
        frame["events"] = int(subset["event"].sum())
        curves.append(frame)

    return pd.concat(curves, ignore_index=True)


def logrank(fleet: pd.DataFrame, group_column: str = "subset") -> dict:
    """Test whether the groups' survival curves differ at all.

    Run before any model is allowed to condition on the group. If the curves
    were indistinguishable, adding subset indicators would be fitting noise.
    """
    result = multivariate_logrank_test(
        fleet["observed_time"],
        fleet[group_column],
        fleet["event"].astype(bool),
    )
    return {
        "test_statistic": float(result.test_statistic),
        "p_value": float(result.p_value),
        "degrees_of_freedom": int(result.degrees_of_freedom),
        "groups": sorted(fleet[group_column].unique().tolist()),
    }


def risk_window_start(risk_set: pd.DataFrame) -> int:
    """The earliest cycle at which any failure has ever been observed.

    Derived from the fit set, never the holdout: it becomes part of the model
    specification, and a cut point chosen with knowledge of the holdout's
    failure times is a leak like any other.
    """
    failures = risk_set.loc[risk_set["failed"] == 1, "cycle"]
    if failures.empty:
        raise ValueError("Cannot locate the risk window: no observed failures.")
    return int(failures.min())


def restrict_to_risk_window(risk_set: pd.DataFrame, start: int) -> pd.DataFrame:
    """Condition the analysis on having survived to ``start``.

    In this fleet no engine fails before cycle 128. Those early cycles are not
    weak evidence of low risk — they are ~40,000 rows in which the outcome is
    constant, and a logistic model asked to explain them drives its intercept
    to minus infinity. That is textbook complete separation: the fit does not
    converge and the Hessian is singular.

    Restricting the risk set is the honest response, and it is a *modelling*
    decision with a name: the hazard is estimated conditional on surviving to
    cycle ``start``, which is delayed entry (left truncation) at a fixed
    landmark. Two things follow, both stated rather than buried:

    * Every quantity the model produces is conditional on the unit having
      reached ``start``. It says nothing about the first 127 cycles, and it
      should not — the data contains no failure there to learn from.
    * Units censored before ``start`` drop out entirely. They contribute no
      information about a window they never entered. ``evaluate.py`` reports
      how many are lost so the reader can judge the cost rather than take it
      on trust.

    Operationally this matches the decision anyway: an engine younger than the
    shortest life ever seen is not a maintenance candidate.
    """
    return risk_set.loc[risk_set["cycle"] >= start].copy()


def derive_band_edges(risk_set: pd.DataFrame, n_bands: int = N_HAZARD_BANDS) -> np.ndarray:
    """Cut points for the piecewise-constant baseline hazard, from failures alone.

    Equal-width bands do not work on this fleet. In the sparse tail several
    20-cycle bands hold only a few dozen rows and no failures at all, and a
    band with zero events has a coefficient of minus infinity — perfect
    separation, singular Hessian, no fit.

    Equal-*event* bands remove the problem by construction. Cut points are the
    quantiles of the observed failure cycles, so every band contains roughly
    the same number of events and none is empty: with ten bands and ~300
    failures in the fit set, about thirty events each.

    Derived from the **fit** risk set only. Quantiles of holdout failure times
    would be the holdout's answers leaking into the design matrix.
    """
    failure_cycles = risk_set.loc[risk_set["failed"] == 1, "cycle"].to_numpy(dtype=float)
    if len(failure_cycles) == 0:
        raise ValueError("Cannot derive hazard bands: the risk set has no failures.")

    quantiles = np.linspace(0.0, 1.0, n_bands + 1)[1:-1]
    interior = np.unique(np.quantile(failure_cycles, quantiles).round())
    return np.concatenate([[failure_cycles.min()], interior])


def assign_band(cycles: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """Band index for each cycle. Band 0 is everything before the first failure."""
    return np.searchsorted(edges, np.asarray(cycles, dtype=float), side="right")


def build_design(risk_set: pd.DataFrame, feature_columns: list[str],
                 edges: np.ndarray | None = None) -> pd.DataFrame:
    """Assemble the design matrix for the discrete-time hazard model.

    Cycle enters as band dummies so the baseline hazard is free to take any
    shape — whether risk accelerates with age is the question a maintenance
    policy turns on, so it is estimated, not imposed by a linear term.

    The **lowest** band is the reference and is left out. Dropping it is not
    optional bookkeeping: after ``restrict_to_risk_window`` every row falls in
    some band, so a full set of dummies would sum to one in every row and be
    exactly collinear with the intercept. The fit would then depend on whatever
    the linear algebra happened to do with a singular Hessian rather than on
    the data. Every band coefficient therefore reads as log-odds of failing
    this cycle relative to the youngest band at which failures occur.
    """
    design = risk_set.loc[:, feature_columns].astype(float).copy()

    if edges is not None:
        bands = assign_band(risk_set["cycle"].to_numpy(), edges)
        for level in range(1, len(edges) + 1):
            if level == BAND_REFERENCE:
                continue
            design[f"band_{level}"] = (bands == level).astype(float)

    return sm.add_constant(design, has_constant="add")


def fit_discrete_hazard(risk_set: pd.DataFrame, feature_columns: list[str],
                        edges: np.ndarray | None = None):
    """Fit the per-cycle failure probability with unit-clustered standard errors.

    The outcome is one Bernoulli trial per cycle at risk. Censored units
    contribute every cycle they were watched with outcome 0 — which is the
    whole point: they are informative about *not* failing, and the complete-case
    alternative throws that away.
    """
    design = build_design(risk_set, feature_columns, edges=edges)
    outcome = risk_set["failed"].astype(int)

    model = sm.Logit(outcome, design)
    return model.fit(
        disp=False,
        cov_type="cluster",
        cov_kwds={"groups": risk_set["unit_id"]},
        maxiter=200,
    )


def fit_complete_case(risk_set: pd.DataFrame, feature_columns: list[str],
                      edges: np.ndarray | None = None):
    """The same model, fitted only on units whose failure was observed.

    This is the censoring-blind alternative, and it is not a straw man: any
    method that needs a completed lifetime as its target — which is what
    "regress on remaining useful life" needs — can only be trained on these
    units. Fitting it explicitly is how the bias gets measured instead of
    asserted.
    """
    observed_only = risk_set.loc[risk_set["event"] == 1]
    return fit_discrete_hazard(observed_only, feature_columns, edges=edges)


def fit_cox_time_varying(risk_set: pd.DataFrame, feature_columns: list[str],
                         penalizer: float = 0.1):
    """Cox PH on the counting-process form, clustered by unit.

    A small ridge penalty is used because several sensor pairs are near-
    duplicates by construction (corrected and uncorrected fan speed, for
    instance, correlate at 0.97). Without it the partial likelihood is close to
    flat along those directions and the fit wanders; with it the model
    converges and the *ranking* it produces — which is all Cox is used for
    here — is stable. The penalty is reported, not hidden: it biases individual
    coefficients toward zero, which is why the coefficient table is read at the
    level of "which sensors matter", not "this one is worth exactly 0.31".
    """
    columns = ["unit_id", "entry", "exit", "failed", *feature_columns]
    frame = risk_set.loc[:, columns].copy()
    frame[feature_columns] = frame[feature_columns].astype(float)

    fitter = CoxTimeVaryingFitter(penalizer=penalizer)
    fitter.fit(
        frame,
        id_col="unit_id",
        event_col="failed",
        start_col="entry",
        stop_col="exit",
        show_progress=False,
    )
    return fitter


def schoenfeld_residuals(fitter: CoxTimeVaryingFitter, risk_set: pd.DataFrame,
                         feature_columns: list[str]) -> pd.DataFrame:
    """Schoenfeld residuals: observed covariate at each failure minus its expectation.

    At each failure time the model implies a distribution over *which* of the
    engines still at risk should be the one to fail — engine *k* with weight
    proportional to ``exp(βᵀx_k)``. The residual is what actually failed minus
    that weighted mean:

        r_j = x_(failed at t_j) − Σ_k w_k x_k ,  w_k ∝ exp(βᵀx_k) over the risk set

    Under proportional hazards these residuals are noise with no relation to
    time. If a covariate's effect strengthens as engines age, its residuals
    trend with time, and that trend is the diagnostic.

    Computing them explicitly matters here. The obvious shortcut — correlating
    the raw covariate values at failures with failure time — tests the wrong
    thing entirely: the drift features *rise with age by construction*, so that
    shortcut flags essentially every covariate as a violation whether or not
    the model's assumption is actually broken. The residual subtracts the
    model's own expectation, which is precisely what removes that artefact.
    """
    coefficients = np.array([float(fitter.params_.get(column, 0.0))
                             for column in feature_columns])
    covariates = risk_set.loc[:, feature_columns].to_numpy(dtype=float)
    entry = risk_set["entry"].to_numpy(dtype=float)
    exit_ = risk_set["exit"].to_numpy(dtype=float)
    failed = risk_set["failed"].to_numpy(dtype=bool)

    linear = covariates @ coefficients
    # Subtract the max before exponentiating: without it a large linear
    # predictor overflows and the weights come out as nan.
    failure_times = np.unique(exit_[failed])

    rows: list[np.ndarray] = []
    kept_times: list[float] = []
    skipped = 0

    for time in failure_times:
        at_risk = (entry < time) & (exit_ >= time)
        # A risk set of one carries no information: the engine that failed is
        # the only candidate, so its residual is identically zero and would
        # dilute the correlation toward "no violation". Skipped and counted,
        # never silently included.
        if at_risk.sum() < 2:
            skipped += int((failed & (exit_ == time)).sum())
            continue
        weights = np.exp(linear[at_risk] - linear[at_risk].max())
        weights /= weights.sum()
        expected = weights @ covariates[at_risk]

        for index in np.flatnonzero(failed & (exit_ == time)):
            rows.append(covariates[index] - expected)
            kept_times.append(time)

    if not rows:
        raise ValueError("No usable failure times; the PH test has nothing to work with.")

    residuals = pd.DataFrame(rows, columns=feature_columns)
    residuals.insert(0, "failure_time", kept_times)
    residuals.attrs["failures_skipped_singleton_risk_set"] = skipped
    residuals.attrs["failures_used"] = len(residuals)
    return residuals


def test_proportional_hazards(fitter: CoxTimeVaryingFitter,
                              risk_set: pd.DataFrame,
                              feature_columns: list[str]) -> pd.DataFrame:
    """Check Cox's central assumption instead of inheriting it.

    Proportional hazards says a covariate's effect on the log-hazard is the
    same at cycle 130 as at cycle 350. For a degrading engine that is a strong
    claim worth testing: a sensor drifting high may matter far more late in
    life than early.

    The test rank-correlates each covariate's Schoenfeld residuals with failure
    time. Under PH the correlation is zero; a significant one means that
    covariate's effect moves with age. Implemented here rather than through
    ``check_assumptions`` because lifelines does not expose that helper for the
    time-varying fitter.

    Rank correlation rather than Pearson, because a handful of extreme
    residuals late in life would otherwise drive the statistic on their own.
    """
    residuals = schoenfeld_residuals(fitter, risk_set, feature_columns)
    rank_times = residuals["failure_time"].rank().to_numpy()
    n = len(residuals)

    rows: list[dict] = []
    for column in feature_columns:
        values = residuals[column].to_numpy(dtype=float)
        if np.std(values) == 0 or n < 5:
            rows.append({"covariate": column, "correlation": np.nan, "p_value": np.nan,
                         "coefficient": float(fitter.params_.get(column, np.nan))})
            continue
        correlation = float(np.corrcoef(pd.Series(values).rank().to_numpy(), rank_times)[0, 1])
        z = np.arctanh(np.clip(correlation, -0.999999, 0.999999)) * np.sqrt(n - 3)
        rows.append({
            "covariate": column,
            "correlation": correlation,
            "p_value": float(2 * (1 - _standard_normal_cdf(abs(z)))),
            "coefficient": float(fitter.params_.get(column, np.nan)),
        })

    result = pd.DataFrame(rows)
    # Seventeen covariates tested at once is seventeen chances to find a
    # spurious violation. Holm-Bonferroni controls the family-wise error rate
    # without assuming the tests are independent — they are not, since the
    # sensors are correlated.
    result = result.sort_values("p_value").reset_index(drop=True)
    n_tests = result["p_value"].notna().sum()
    adjusted = []
    running = 0.0
    for position, p_value in enumerate(result["p_value"]):
        if pd.isna(p_value):
            adjusted.append(np.nan)
            continue
        running = max(running, min(1.0, p_value * (n_tests - position)))
        adjusted.append(running)
    result["p_holm"] = adjusted
    result["violates_ph"] = result["p_holm"] < 0.05
    return result


def _standard_normal_cdf(x: float) -> float:
    """Normal CDF via the error function, so scipy is not needed for one call."""
    from math import erf, sqrt

    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def predict_hazard(result, risk_set: pd.DataFrame, feature_columns: list[str],
                   edges: np.ndarray | None = None) -> np.ndarray:
    """Per-cycle failure probability for each row of a risk set."""
    design = build_design(risk_set, feature_columns, edges=edges)
    design = design.reindex(columns=result.params.index, fill_value=0.0)
    return np.asarray(result.predict(design), dtype=float)


def horizon_risk(result, risk_set: pd.DataFrame, feature_columns: list[str],
                 edges: np.ndarray, horizon: int = DEFAULT_HORIZON) -> np.ndarray:
    """P(fail within ``horizon`` cycles | alive now), per row.

    Built from the per-cycle hazard by compounding survival across the horizon:
    the unit survives the horizon only if it survives every cycle in it, so the
    risk is one minus the product of per-cycle survivals.

    Covariates are frozen at their current value (argued in the module
    docstring) but the **cycle band advances**, so the model's estimated ageing
    is carried forward even though the sensor readings are not. A unit that
    would cross into a higher-hazard band during the horizon has that reflected
    in its risk.
    """
    design = build_design(risk_set, feature_columns, edges=edges)
    design = design.reindex(columns=result.params.index, fill_value=0.0)

    band_columns = [column for column in design.columns if column.startswith("band_")]
    band_index = {column: design.columns.get_loc(column) for column in band_columns}
    band_levels = {column: int(column.split("_")[1]) for column in band_columns}

    coefficients = result.params.to_numpy()
    base = design.to_numpy(dtype=float).copy()
    current_cycle = risk_set["cycle"].to_numpy(dtype=float)
    survival = np.ones(len(design), dtype=float)

    for step in range(1, horizon + 1):
        future_band = assign_band(current_cycle + step, edges)
        for column, level in band_levels.items():
            base[:, band_index[column]] = (future_band == level).astype(float)
        hazard = 1.0 / (1.0 + np.exp(-(base @ coefficients)))
        survival *= (1.0 - hazard)

    return 1.0 - survival


def landmark_frame(risk_set: pd.DataFrame, landmark: int) -> pd.DataFrame:
    """The units still at risk at ``landmark``, with their readings at that cycle.

    This is what makes a time-varying model scoreable against a plain survival
    metric. Fix a cycle; keep the units that have neither failed nor been
    censored by then; take each one's covariates at exactly that cycle; and
    define the outcome as time from the landmark to failure or censoring.

    Units already gone by the landmark are excluded, not treated as failures.
    They are not part of the population the question is about — an operator
    standing at cycle 100 is deciding about engines that are running at cycle
    100.
    """
    at_landmark = risk_set.loc[risk_set["cycle"] == landmark].copy()
    if at_landmark.empty:
        raise ValueError(f"No units are at risk at cycle {landmark}.")

    at_landmark["time_from_landmark"] = at_landmark["observed_time"] - landmark
    at_landmark["event_from_landmark"] = at_landmark["event"].astype(int)
    return at_landmark.reset_index(drop=True)


def fit_random_survival_forest(landmark_data: pd.DataFrame,
                               feature_columns: list[str],
                               n_estimators: int = 300,
                               seed: int = RANDOM_SEED):
    """A flexible comparator, to bound what the interpretable model gives up.

    Its job is not to be deployed. If the forest is barely ahead, the
    discrete-time model's interpretability is free and the project says so; if
    it is far ahead, the linear form is costing something real and the project
    says that instead. Either way the number is reported rather than assumed —
    the same move the breast-cancer project made when nine classifiers turned
    out to sit inside one standard deviation of each other.
    """
    from sksurv.ensemble import RandomSurvivalForest
    from sksurv.util import Surv

    y = Surv.from_arrays(
        event=landmark_data["event_from_landmark"].astype(bool).to_numpy(),
        time=landmark_data["time_from_landmark"].astype(float).to_numpy(),
    )
    X = landmark_data.loc[:, feature_columns].astype(float).to_numpy()

    forest = RandomSurvivalForest(
        n_estimators=n_estimators,
        min_samples_leaf=15,
        max_features="sqrt",
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(X, y)
    return forest


def tidy(result) -> pd.DataFrame:
    """Coefficients as odds ratios with clustered confidence intervals.

    Odds ratios because the readable statement is "a unit one standard
    deviation further from its own baseline on sensor 11 is X times as likely
    to fail this cycle", not "the log-odds rise by 0.24".
    """
    confidence = result.conf_int()
    frame = pd.DataFrame({
        "coefficient": result.params,
        "std_error": result.bse,
        "p_value": result.pvalues,
        "odds_ratio": np.exp(result.params),
        "or_lower": np.exp(confidence[0]),
        "or_upper": np.exp(confidence[1]),
    })
    frame.index.name = "term"
    return frame.reset_index()
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.survival'] = _m
setattr(_src, 'survival', _m)
exec(compile(_code_survival, 'src/survival.py', 'exec'), _m.__dict__)

# ---- src/evaluate.py ------------------------------------------------
_m = types.ModuleType('src.evaluate')
_m.__dict__['__name__'] = 'src.evaluate'
_m.__dict__['__file__'] = _ROOT + '/src/evaluate.py'
_m.__dict__['__package__'] = 'src'
_code_evaluate = r'''"""Score the models, and measure the cost of ignoring censoring.

Two jobs, and the second is the one the project is really about.

Job one: does the model discriminate and is it calibrated
---------------------------------------------------------
Discrimination and calibration are different questions and a maintenance
policy needs both. Discrimination asks whether the model *orders* engines
correctly — is the one it thinks is sicker actually the one that fails first?
Calibration asks whether the number means what it says — of the engines it
gives a 10% chance of failing within 30 cycles, do about 10% fail?

A model can discriminate perfectly and be badly calibrated, and for this
project that would be fatal: the whole decision layer is "act when the
probability crosses a threshold", so a probability that is systematically
double the truth moves the threshold to the wrong place. Both are reported.

Why two concordance indices
---------------------------
**Harrell's C** is the familiar one, and it is biased under censoring: the
comparable pairs it can form are not a random sample of all pairs, and the
heavier the censoring the worse the distortion. **Uno's C** reweights pairs by
the inverse probability of remaining uncensored, which removes that dependence.

Both are reported precisely because the gap between them is informative. In a
project whose subject is censoring, quoting only the censoring-sensitive metric
would be self-defeating.

Scoring is done at **landmarks**. A time-varying-covariate model does not have
one C-index; it has one per moment at which you might ask. Fixing a cycle,
keeping the units still running then, and scoring the ordering of what happens
next is both well defined and exactly the operational question: standing at
cycle 150, which of these engines should I worry about?

Job two: what does dropping censored units actually cost
---------------------------------------------------------
This is the measurement the plan is built around. Three estimates of the same
truth are compared:

1. **Truth** — computed on the full run-to-failure trajectories. Available only
   because the raw benchmark has them; no real fleet snapshot could produce it.
2. **Complete-case** — censored units deleted. This is what any method needing
   a completed lifetime as its target is forced to do, "regress on remaining
   useful life" included.
3. **Censoring-aware** — the survival estimate, which keeps them.

The complete-case estimate is not just noisier. A unit is censored exactly when
its lifetime exceeds its snapshot horizon, so deleting censored units deletes
the long-lived ones preferentially. The surviving sample is biased short, and
the fleet looks more fragile than it is. ``censoring_bias`` puts a number on it
in cycles, and ``bias_by_censoring_rate`` shows the number growing as the fleet
gets more realistic — which is the direction that matters, because real fleets
are mostly survivors.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.util import Surv

from src import survival

N_BOOTSTRAP = 400
RANDOM_SEED = 20260801

# Bins used to check calibration. Quantile bins rather than fixed-width,
# because predicted risks are heavily concentrated near zero and fixed-width
# bins would put almost every engine in the first one and measure nothing.
N_CALIBRATION_BINS = 10


def _surv_array(times: np.ndarray, events: np.ndarray):
    """scikit-survival's structured (event, time) array."""
    return Surv.from_arrays(event=np.asarray(events).astype(bool),
                            time=np.asarray(times).astype(float))


def landmark_scores(result, risk_set: pd.DataFrame, feature_columns: list[str],
                    edges: np.ndarray, fit_risk_set: pd.DataFrame,
                    landmarks: tuple[int, ...] = survival.LANDMARK_CYCLES,
                    horizon: int = survival.DEFAULT_HORIZON) -> pd.DataFrame:
    """Discrimination at each landmark, by both concordance indices.

    Uno's C needs the censoring distribution, and it must be estimated on the
    **fit** data: using the evaluation set's own censoring pattern to weight
    the evaluation set is circular. It also requires the evaluation times to
    lie inside the range the fit data covers, so times beyond that are trimmed
    and the number trimmed is reported rather than quietly dropped.
    """
    rows: list[dict] = []

    for landmark in landmarks:
        try:
            evaluation = survival.landmark_frame(risk_set, landmark)
        except ValueError:
            continue
        if evaluation["event_from_landmark"].sum() < 5:
            continue

        risk = survival.horizon_risk(result, evaluation, feature_columns, edges,
                                     horizon=horizon)
        times = evaluation["time_from_landmark"].to_numpy(dtype=float)
        events = evaluation["event_from_landmark"].to_numpy(dtype=bool)

        harrell = concordance_index_censored(events, times, risk)[0]

        try:
            fit_landmark = survival.landmark_frame(fit_risk_set, landmark)
            train_surv = _surv_array(
                fit_landmark["time_from_landmark"].to_numpy(),
                fit_landmark["event_from_landmark"].to_numpy(),
            )
            tau = min(times[events].max(),
                      fit_landmark.loc[fit_landmark["event_from_landmark"] == 1,
                                       "time_from_landmark"].max())
            uno = concordance_index_ipcw(
                train_surv, _surv_array(times, events), risk, tau=float(tau)
            )[0]
        except (ValueError, ZeroDivisionError):
            uno = np.nan

        rows.append({
            "landmark": landmark,
            "units_at_risk": len(evaluation),
            "events": int(events.sum()),
            "harrell_c": float(harrell),
            "uno_c": float(uno),
            "mean_predicted_risk": float(risk.mean()),
        })

    return pd.DataFrame(rows)


def time_dependent_auc(result, risk_set: pd.DataFrame, feature_columns: list[str],
                       edges: np.ndarray, fit_risk_set: pd.DataFrame,
                       landmark: int, horizon: int = survival.DEFAULT_HORIZON,
                       n_points: int = 8) -> pd.DataFrame:
    """AUC as a function of how far ahead you are asking.

    One number for "does the model work" hides the thing a maintenance planner
    cares about most: models are usually good at spotting imminent failure and
    much worse at spotting it early, and early is when action is still cheap.
    The shape of this curve is the honest statement of how much warning the
    model really provides.
    """
    evaluation = survival.landmark_frame(risk_set, landmark)
    fit_landmark = survival.landmark_frame(fit_risk_set, landmark)

    risk = survival.horizon_risk(result, evaluation, feature_columns, edges,
                                 horizon=horizon)
    times = evaluation["time_from_landmark"].to_numpy(dtype=float)
    events = evaluation["event_from_landmark"].to_numpy(dtype=bool)

    event_times = np.sort(times[events])
    if len(event_times) < n_points + 2:
        n_points = max(2, len(event_times) - 2)
    grid = np.unique(
        np.quantile(event_times, np.linspace(0.1, 0.9, n_points)).round()
    )

    auc, mean_auc = cumulative_dynamic_auc(
        _surv_array(fit_landmark["time_from_landmark"].to_numpy(),
                    fit_landmark["event_from_landmark"].to_numpy()),
        _surv_array(times, events),
        risk,
        times=grid,
    )
    return pd.DataFrame({
        "cycles_ahead": grid,
        "auc": auc,
        "mean_auc": mean_auc,
        "landmark": landmark,
    })


def brier(result, risk_set: pd.DataFrame, feature_columns: list[str],
          edges: np.ndarray, fit_risk_set: pd.DataFrame,
          landmark: int, horizons: tuple[int, ...] = (10, 20, 30, 40, 50)) -> dict:
    """Integrated Brier score: accuracy of the *probability*, not just the ranking.

    The Brier score is squared error on a probability, so unlike the C-index it
    punishes a model that orders engines correctly while stating the wrong
    number. Integrated over horizons it summarises calibration and
    discrimination together. Lower is better; 0.25 is what a model that always
    predicts 0.5 would score, and the always-predict-the-KM-curve baseline is
    reported alongside so the number has something to mean.
    """
    evaluation = survival.landmark_frame(risk_set, landmark)
    fit_landmark = survival.landmark_frame(fit_risk_set, landmark)

    times = evaluation["time_from_landmark"].to_numpy(dtype=float)
    events = evaluation["event_from_landmark"].to_numpy(dtype=bool)

    grid = np.array([h for h in horizons if h < times[events].max()], dtype=float)
    if len(grid) < 2:
        return {"integrated_brier_score": np.nan, "baseline_integrated_brier_score": np.nan}

    # Survival probability at each horizon, per unit.
    predictions = np.column_stack([
        1.0 - survival.horizon_risk(result, evaluation, feature_columns, edges,
                                    horizon=int(h))
        for h in grid
    ])

    train = _surv_array(fit_landmark["time_from_landmark"].to_numpy(),
                        fit_landmark["event_from_landmark"].to_numpy())
    test = _surv_array(times, events)

    score = integrated_brier_score(train, test, predictions, grid)

    # The reference every prognostic model should have to beat: the fleet-wide
    # Kaplan-Meier curve, which uses no sensor information at all. A model that
    # cannot beat it is not reading the sensors to any useful effect.
    fitter = KaplanMeierFitter().fit(
        fit_landmark["time_from_landmark"],
        event_observed=fit_landmark["event_from_landmark"].astype(bool),
    )
    baseline_curve = np.asarray(
        [float(fitter.predict(h)) for h in grid], dtype=float
    )
    baseline = integrated_brier_score(
        train, test, np.tile(baseline_curve, (len(times), 1)), grid
    )

    return {
        "landmark": landmark,
        "integrated_brier_score": float(score),
        "baseline_integrated_brier_score": float(baseline),
        "improvement": float(baseline - score),
        "horizons": grid.tolist(),
    }


def calibration(result, risk_set: pd.DataFrame, feature_columns: list[str],
                edges: np.ndarray, landmark: int,
                horizon: int = survival.DEFAULT_HORIZON,
                n_bins: int = N_CALIBRATION_BINS) -> pd.DataFrame:
    """Predicted vs observed failure rate, in bins of predicted risk.

    The observed rate is a **Kaplan-Meier** estimate inside each bin, not a raw
    proportion. Using a raw proportion would treat a unit censored at cycle 10
    of a 30-cycle horizon as a survivor, which understates the observed rate and
    would make the model look systematically over-cautious — the exact error
    this project exists to avoid making.
    """
    evaluation = survival.landmark_frame(risk_set, landmark)
    risk = survival.horizon_risk(result, evaluation, feature_columns, edges,
                                 horizon=horizon)

    evaluation = evaluation.assign(predicted=risk)
    # Bins can collapse when many units share a near-identical low risk; `duplicates`
    # keeps that from raising rather than silently mislabelling.
    evaluation["bin"] = pd.qcut(
        evaluation["predicted"], q=n_bins, labels=False, duplicates="drop"
    )

    rows: list[dict] = []
    for bin_index, group in evaluation.groupby("bin", observed=True):
        fitter = KaplanMeierFitter().fit(
            group["time_from_landmark"],
            event_observed=group["event_from_landmark"].astype(bool),
        )
        observed = 1.0 - float(fitter.predict(horizon))
        rows.append({
            "bin": int(bin_index),
            "units": len(group),
            "mean_predicted": float(group["predicted"].mean()),
            "observed_km": observed,
        })

    return pd.DataFrame(rows)


def censoring_bias(fleet: pd.DataFrame) -> dict:
    """The headline of the censoring critique, in cycles.

    Three estimates of mean life, plus the Kaplan-Meier median, which is the
    quantity that actually survives censoring cleanly:

    ``truth``          mean of every unit's true lifetime — unavailable in real life
    ``complete_case``  mean lifetime of units whose failure was observed
    ``naive_observed`` mean of the observed times, counting a censored unit's
                       snapshot as if it were its lifetime — the other common
                       error, and it biases the opposite way

    The complete-case gap is the number the README leads with, because it
    quantifies what "just drop the units that have not failed yet" costs.
    """
    truth = float(fleet["true_lifetime"].mean())
    observed = fleet.loc[fleet["event"] == 1, "true_lifetime"]
    complete_case = float(observed.mean())
    naive = float(fleet["observed_time"].mean())

    fitter = KaplanMeierFitter().fit(
        fleet["observed_time"], event_observed=fleet["event"].astype(bool)
    )
    km_median = float(fitter.median_survival_time_)
    true_median = float(fleet["true_lifetime"].median())

    return {
        "units": int(len(fleet)),
        "censored_fraction": float(fleet["censored"].mean()),
        "true_mean_life": truth,
        "complete_case_mean_life": complete_case,
        "complete_case_bias_cycles": complete_case - truth,
        "complete_case_bias_pct": 100.0 * (complete_case - truth) / truth,
        "naive_observed_mean_life": naive,
        "naive_bias_cycles": naive - truth,
        "true_median_life": true_median,
        "kaplan_meier_median": km_median,
        "kaplan_meier_median_error": km_median - true_median,
    }


def bias_by_censoring_rate(trajectories: pd.DataFrame,
                           rates: tuple[float, ...] = (0.2, 0.3, 0.4, 0.5, 0.6, 0.7),
                           seed: int = RANDOM_SEED) -> pd.DataFrame:
    """How the complete-case bias grows as the fleet gets more realistic.

    A single censoring rate could always be dismissed as a contrived setting.
    Sweeping it answers the objection directly: the bias is not an artefact of
    one choice, it is monotone in how much of the fleet is still running — and
    real fleets are mostly still running.

    Kaplan-Meier's error is reported in the same table as the control. It stays
    near zero across the sweep, which is the positive half of the argument:
    the information is not lost, it is only lost by the wrong estimator.
    """
    from src.panel import induce_censoring, unit_lifetimes

    lifetimes = unit_lifetimes(trajectories)
    rows: list[dict] = []

    for rate in rates:
        fleet = induce_censoring(lifetimes, target_rate=rate, seed=seed)
        summary = censoring_bias(fleet)
        rows.append({
            "target_censor_rate": rate,
            "realised_censor_rate": summary["censored_fraction"],
            "complete_case_bias_cycles": summary["complete_case_bias_cycles"],
            "complete_case_bias_pct": summary["complete_case_bias_pct"],
            "kaplan_meier_median_error": summary["kaplan_meier_median_error"],
            "events": int(fleet["event"].sum()),
        })

    return pd.DataFrame(rows)


def compare_feature_sets(fit_risk_set: pd.DataFrame, holdout_risk_set: pd.DataFrame,
                         drift_columns: list[str], level_columns: list[str],
                         edges: np.ndarray,
                         landmarks: tuple[int, ...] = survival.LANDMARK_CYCLES,
                         horizon: int = survival.DEFAULT_HORIZON) -> pd.DataFrame:
    """Test the 'drift, not level' claim instead of only arguing it.

    ``panel.add_dynamics`` argues that a sensor's *level* confounds an engine
    that started worn with one that has worn out, and that the change from the
    unit's own baseline separates them. That is a claim about out-of-sample
    behaviour, so it is checked out of sample: fit all three feature sets and
    compare in-sample fit against holdout discrimination.
    """
    rows: list[dict] = []
    candidates = {
        "drift": drift_columns,
        "level": level_columns,
        "drift + level": drift_columns + level_columns,
    }

    for name, columns in candidates.items():
        result = survival.fit_discrete_hazard(fit_risk_set, columns, edges)
        scores = landmark_scores(result, holdout_risk_set, columns, edges,
                                 fit_risk_set, landmarks=landmarks, horizon=horizon)
        rows.append({
            "feature_set": name,
            "n_parameters": int(len(result.params)),
            "in_sample_pseudo_r2": float(result.prsquared),
            "holdout_harrell_c": float(scores["harrell_c"].mean()),
            "holdout_uno_c": float(scores["uno_c"].mean()),
        })

    return pd.DataFrame(rows)


def bootstrap_metric(values: np.ndarray, groups: np.ndarray,
                     statistic, n_boot: int = N_BOOTSTRAP,
                     seed: int = RANDOM_SEED) -> tuple[float, float, float]:
    """Percentile confidence interval, resampling **units** rather than rows.

    Resampling rows would treat a unit's hundreds of correlated cycle
    observations as independent draws and produce an interval several times too
    narrow. The unit is the independent sampling unit here, so the unit is what
    gets resampled — the same reasoning that makes the standard errors
    clustered.
    """
    generator = np.random.default_rng(seed)
    unique_groups = np.unique(groups)
    estimates: list[float] = []

    for _ in range(n_boot):
        chosen = generator.choice(unique_groups, size=len(unique_groups), replace=True)
        index = np.concatenate([np.flatnonzero(groups == group) for group in chosen])
        try:
            estimates.append(float(statistic(values[index], index)))
        except (ValueError, ZeroDivisionError):
            continue

    if not estimates:
        return np.nan, np.nan, np.nan

    array = np.asarray(estimates)
    return (
        float(np.mean(array)),
        float(np.percentile(array, 2.5)),
        float(np.percentile(array, 97.5)),
    )
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.evaluate'] = _m
setattr(_src, 'evaluate', _m)
exec(compile(_code_evaluate, 'src/evaluate.py', 'exec'), _m.__dict__)

# ---- src/figures.py -------------------------------------------------
_m = types.ModuleType('src.figures')
_m.__dict__['__name__'] = 'src.figures'
_m.__dict__['__file__'] = _ROOT + '/src/figures.py'
_m.__dict__['__package__'] = 'src'
_code_figures = r'''"""Figures for the findings.

One function per figure, each taking already-computed results rather than
refitting anything. That keeps the plotting free of analysis: if a number in a
chart disagrees with a number in the README, the cause is in the module that
produced it, not here.

House style is set once in ``apply_style`` — no per-chart colour choices, so
the figures read as one set, and as one set with the other projects in the
portfolio.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # No display in a scripted run; must be set before pyplot.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

# One deep tone for the thing being argued for, one muted tone for what it is
# argued against, one warm accent for the decision. Distinguishable in
# greyscale and to the most common forms of colour blindness.
PRIMARY = "#1F3A5F"
MUTED = "#B4B8C5"
ACCENT = "#C4622D"
POSITIVE = "#3E7D5A"
GRID = "#E3E5EA"


def apply_style() -> None:
    plt.rcParams.update({
        "figure.dpi": 130,
        "savefig.dpi": 130,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#3C4048",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "legend.frameon": False,
    })


def _save(fig: plt.Figure, name: str) -> Path:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / name
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def degradation_trajectories(risk_set: pd.DataFrame, sensor: str = "sensor_11",
                             n_units: int = 60, seed: int = 0) -> Path:
    """Why a survival model is needed at all: wear is gradual and unit-specific.

    Each line is one engine's drift on the most degradation-correlated sensor,
    against its age. They rise, but they rise from different starting points at
    different rates — which is the argument for a model that reads the unit's
    own trajectory rather than one that reads the calendar.
    """
    apply_style()
    generator = np.random.default_rng(seed)
    units = generator.choice(risk_set["unit_id"].unique(), size=n_units, replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    for unit in units:
        frame = risk_set.loc[risk_set["unit_id"] == unit]
        failed = frame["event"].iloc[0] == 1
        axes[0].plot(frame["cycle"], frame[f"{sensor}_drift"],
                     color=ACCENT if failed else MUTED,
                     alpha=0.55 if failed else 0.45, linewidth=0.9)

    axes[0].set_xlabel("Cycles in service")
    axes[0].set_ylabel(f"{sensor} drift from own baseline (sd)")
    axes[0].set_title("Every engine wears differently")
    axes[0].plot([], [], color=ACCENT, label="failure observed")
    axes[0].plot([], [], color=MUTED, label="still running at snapshot")
    axes[0].legend(loc="upper left")

    # The same signal aligned on remaining life, which is what makes the
    # degradation obvious — and what an operator cannot do, because the
    # x-axis is only knowable in hindsight.
    for unit in units:
        frame = risk_set.loc[risk_set["unit_id"] == unit]
        if frame["event"].iloc[0] != 1:
            continue
        remaining = frame["observed_time"].iloc[0] - frame["cycle"]
        axes[1].plot(-remaining, frame[f"{sensor}_drift"],
                     color=ACCENT, alpha=0.4, linewidth=0.9)

    axes[1].set_xlabel("Cycles until failure")
    axes[1].set_ylabel(f"{sensor} drift from own baseline (sd)")
    axes[1].set_title("Aligned on failure — the signal an operator cannot see")

    return _save(fig, "01_degradation_trajectories.png")


def survival_curves(curves: pd.DataFrame) -> Path:
    """Kaplan-Meier by subset, with confidence bands.

    Included before any model, to establish that the four fleet designs really
    do have different survival — the log-rank test in the same section is what
    licenses conditioning on subset later.
    """
    apply_style()
    fig, axis = plt.subplots(figsize=(7.5, 4.6))
    palette = [PRIMARY, ACCENT, POSITIVE, "#8C6BB1"]

    for colour, (group, frame) in zip(palette, curves.groupby("group")):
        axis.step(frame["cycle"], frame["survival"], where="post",
                  color=colour, linewidth=1.8, label=f"{group} (n={frame['units'].iloc[0]})")
        axis.fill_between(frame["cycle"], frame["lower"], frame["upper"],
                          step="post", color=colour, alpha=0.12)

    axis.set_xlabel("Cycles in service")
    axis.set_ylabel("Probability still running")
    axis.set_title("Survival differs by operating condition and fault mode")
    axis.set_ylim(0, 1.02)
    axis.legend(loc="lower left")
    return _save(fig, "02_survival_by_subset.png")


def censoring_bias(bias_sweep: pd.DataFrame) -> Path:
    """The central claim, as a picture: the bias grows with how realistic the fleet is.

    Two series on purpose. The complete-case error walks away from zero as more
    of the fleet is still running; the Kaplan-Meier error stays on it. That
    pairing is the whole argument — the information is not destroyed by
    censoring, it is destroyed by the wrong estimator.
    """
    apply_style()
    fig, axis = plt.subplots(figsize=(7.5, 4.6))

    rate = 100 * bias_sweep["realised_censor_rate"]
    axis.axhline(0, color="#3C4048", linewidth=1.0)
    axis.plot(rate, bias_sweep["complete_case_bias_cycles"], marker="o",
              color=ACCENT, linewidth=2.0,
              label="Complete-case mean life\n(drop unfailed engines)")
    axis.plot(rate, bias_sweep["kaplan_meier_median_error"], marker="s",
              color=PRIMARY, linewidth=2.0,
              label="Kaplan-Meier median error\n(censoring-aware)")

    axis.set_xlabel("Share of the fleet still running at the snapshot (%)")
    axis.set_ylabel("Error vs the known truth (cycles)")
    axis.set_title("Dropping unfailed engines biases life estimates; survival methods do not")
    axis.legend(loc="lower left")
    return _save(fig, "03_censoring_bias.png")


def baseline_hazard(coefficients: pd.DataFrame, edges: np.ndarray) -> Path:
    """The estimated shape of ageing — the thing a linear term would have assumed."""
    apply_style()
    bands = coefficients.loc[coefficients["term"].str.startswith("band_")].copy()
    bands["level"] = bands["term"].str.split("_").str[1].astype(int)
    bands = bands.sort_values("level")
    bands["cycle"] = [edges[min(level - 1, len(edges) - 1)] for level in bands["level"]]

    fig, axis = plt.subplots(figsize=(7.5, 4.6))
    axis.axhline(1.0, color="#3C4048", linewidth=1.0)
    axis.plot(bands["cycle"], bands["odds_ratio"], marker="o", color=PRIMARY, linewidth=2.0)
    axis.fill_between(bands["cycle"], bands["or_lower"], bands["or_upper"],
                      color=PRIMARY, alpha=0.15)

    axis.set_yscale("log")
    axis.set_xlabel("Cycles in service (band lower edge)")
    axis.set_ylabel("Odds of failing this cycle\n(relative to youngest band)")
    axis.set_title("The hazard was estimated, not assumed")
    return _save(fig, "04_baseline_hazard.png")


def calibration(frame: pd.DataFrame, landmark: int) -> Path:
    """Predicted against observed — the diagnostic that drives the policy design.

    The departure from the diagonal is the reason ``policy.py`` sweeps the
    threshold instead of computing it. Plotting it prominently rather than
    burying it is the point: it is the project's main weakness and it changes
    what the project does next.
    """
    apply_style()
    fig, axis = plt.subplots(figsize=(6.2, 5.4))

    limit = max(frame["mean_predicted"].max(), frame["observed_km"].max()) * 1.08
    axis.plot([0, limit], [0, limit], color=MUTED, linewidth=1.4, linestyle="--",
              label="perfect calibration")
    axis.scatter(frame["mean_predicted"], frame["observed_km"],
                 s=frame["units"] * 4, color=ACCENT, alpha=0.85, zorder=3,
                 label="risk decile (area = engines)")

    axis.set_xlabel(f"Mean predicted {30}-cycle failure risk")
    axis.set_ylabel("Observed failure rate (Kaplan-Meier)")
    axis.set_title(f"Risk is under-stated: predictions sit below the line (cycle {landmark})")
    axis.legend(loc="upper left")
    return _save(fig, "05_calibration.png")


def threshold_sweep(sweep: pd.DataFrame, optimum: dict, bootstrap: dict,
                    interval_best: float, run_to_failure_cost: float) -> Path:
    """The decision, as one picture: where to put the alarm and what it is worth."""
    apply_style()
    fig, axis = plt.subplots(figsize=(8.0, 4.8))

    axis.plot(sweep["threshold"], sweep["cost_per_1000_cycles"],
              color=PRIMARY, linewidth=2.0, label="Model policy")
    axis.axhline(interval_best, color=ACCENT, linewidth=1.6, linestyle="--",
                 label="Best fixed-interval policy")
    axis.axhline(run_to_failure_cost, color=MUTED, linewidth=1.6, linestyle=":",
                 label="Run to failure")

    low, high = bootstrap["threshold_ci"]
    axis.axvspan(low, high, color=POSITIVE, alpha=0.15,
                 label="95% CI on the optimum")
    axis.axvline(optimum["threshold"], color=POSITIVE, linewidth=1.8)

    axis.set_xscale("log")
    axis.set_xlabel("Alarm threshold (predicted 30-cycle failure risk)")
    axis.set_ylabel("Cost per 1,000 cycles of service")
    axis.set_title("Where to set the alarm")
    axis.legend(loc="upper left")
    return _save(fig, "06_threshold_sweep.png")


def policy_frontier(model_sweep: pd.DataFrame, interval_sweep: pd.DataFrame,
                    optimum: dict) -> Path:
    """Service delivered against failures accepted — the trade the policy is making.

    The clearest single statement of the finding: at any failure rate the
    calendar policy can reach, the condition-based policy delivers more service
    per engine.
    """
    apply_style()
    fig, axis = plt.subplots(figsize=(7.5, 4.8))

    axis.plot(100 * model_sweep["failure_rate"], model_sweep["mean_service_cycles"],
              color=PRIMARY, linewidth=2.0, marker="", label="Model policy")
    axis.plot(100 * interval_sweep["failure_rate"], interval_sweep["mean_service_cycles"],
              color=ACCENT, linewidth=2.0, linestyle="--", label="Fixed-interval policy")
    axis.scatter([100 * optimum["failure_rate"]], [optimum["mean_service_cycles"]],
                 s=110, color=POSITIVE, zorder=4, label="Chosen operating point")

    axis.set_xlabel("In-service failures (% of engines)")
    axis.set_ylabel("Mean cycles of service per engine")
    axis.set_title("Same safety, more service")
    axis.legend(loc="lower right")
    return _save(fig, "07_policy_frontier.png")


def lead_times(distribution: pd.DataFrame, threshold: float) -> Path:
    """How much warning the alarm gives — the operational feasibility check."""
    apply_style()
    pulled = distribution.loc[distribution["pulled"], "lead_time"]

    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    axis.hist(pulled, bins=np.arange(0, pulled.max() + 5, 5),
              color=PRIMARY, alpha=0.85, edgecolor="white")
    axis.axvline(pulled.median(), color=ACCENT, linewidth=2.0,
                 label=f"median {pulled.median():.0f} cycles")
    axis.axvline(np.percentile(pulled, 10), color=MUTED, linewidth=2.0, linestyle="--",
                 label=f"10th percentile {np.percentile(pulled, 10):.0f} cycles")

    axis.set_xlabel("Cycles of warning before the engine would have failed")
    axis.set_ylabel("Engines")
    axis.set_title(f"Warning delivered at the chosen threshold ({threshold:.3f})")
    axis.legend()
    return _save(fig, "08_lead_times.png")


def cost_ratio_sensitivity(frame: pd.DataFrame) -> Path:
    """The recommendation as a curve, so a reader can use their own cost ratio."""
    apply_style()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    axes[0].plot(frame["cost_ratio"], frame["optimal_threshold"],
                 marker="o", color=PRIMARY, linewidth=2.0)
    axes[0].set_xlabel("Failure cost ÷ planned-removal cost")
    axes[0].set_ylabel("Cost-optimal alarm threshold")
    axes[0].set_title("The threshold barely moves")

    axes[1].plot(frame["cost_ratio"], frame["saving_vs_interval_pct"],
                 marker="o", color=POSITIVE, linewidth=2.0)
    axes[1].set_xlabel("Failure cost ÷ planned-removal cost")
    axes[1].set_ylabel("Saving vs best fixed interval (%)")
    axes[1].set_title("The saving holds across the range")
    axes[1].set_ylim(0, max(40, frame["saving_vs_interval_pct"].max() * 1.15))

    return _save(fig, "09_cost_ratio_sensitivity.png")


def time_dependent_auc(frame: pd.DataFrame) -> Path:
    """How far ahead the model can actually see."""
    apply_style()
    fig, axis = plt.subplots(figsize=(7.2, 4.4))
    axis.plot(frame["cycles_ahead"], frame["auc"], marker="o",
              color=PRIMARY, linewidth=2.0)
    axis.axhline(0.5, color=MUTED, linewidth=1.2, linestyle="--", label="no better than chance")
    axis.set_xlabel("Cycles ahead of the landmark")
    axis.set_ylabel("Time-dependent AUC")
    axis.set_ylim(0.4, 1.02)
    axis.set_title("Discrimination decays with how far ahead you ask")
    axis.legend(loc="lower left")
    return _save(fig, "10_time_dependent_auc.png")
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.figures'] = _m
setattr(_src, 'figures', _m)
exec(compile(_code_figures, 'src/figures.py', 'exec'), _m.__dict__)

# ---- src/policy.py --------------------------------------------------
_m = types.ModuleType('src.policy')
_m.__dict__['__name__'] = 'src.policy'
_m.__dict__['__file__'] = _ROOT + '/src/policy.py'
_m.__dict__['__package__'] = 'src'
_code_policy = r'''"""When to pull an engine — turning a probability into a decision.

Everything before this module produces a number. This module spends it.

The decision
------------
At each cycle an operator can leave the engine in service or pull it for
planned maintenance. Pulling early is cheap but wastes remaining life. Leaving
it in risks an in-service failure, which costs several times a planned removal
because of the collateral damage, the unscheduled downtime and the disruption
to everything else on the schedule.

So the policy is a single number — an alarm threshold on predicted risk — and
the question is where to put it.

Why cost per cycle of service, and not cost per unit
----------------------------------------------------
This is the choice that makes the whole exercise honest, and getting it wrong
produces a policy that looks brilliant and is useless.

Score a policy on cost per *unit* and the optimum is degenerate: pull every
engine on its first cycle. Nothing ever fails, every removal is the cheap kind,
and the fleet does no work. The metric has to charge for the service given up,
so the policy is scored as

    cost per 1,000 cycles of service delivered
        = 1000 × Σ cost_i / Σ cycles_delivered_i

Pulling early now buys a lower numerator at the price of a smaller denominator,
which is the real trade. The metric cannot be gamed by being conservative.

For each engine exactly one of two things happens:

* **Pulled at cycle p** (the alarm fired while it was still running) — cost
  ``c_pull``, service ``p`` cycles.
* **Never pulled** — it fails at its true lifetime ``T`` — cost ``c_fail``,
  service ``T`` cycles.

Why the threshold is swept and not calculated
----------------------------------------------
With a perfectly calibrated model the answer is analytic: act when the expected
cost of waiting exceeds the cost of acting, which puts the threshold at
``c_pull / c_fail``. That formula is in every textbook and it is wrong here —
not because the arithmetic fails, but because the model's probabilities are not
calibrated. ``evaluate.calibration`` shows them understating risk badly in the
top bins, and the mechanism is known: ``survival.horizon_risk`` freezes the
sensor readings at their current value, so an engine that is degrading fast has
its future understated.

Two responses are available. Recalibrate the probabilities — but the outcome
needed to fit a recalibration is itself censored, so that repairs one bias by
introducing another. Or stop relying on the probabilities being right: sweep
the threshold and pick the one that actually costs least. The threshold is then
a *rank* cut, and the model only has to order engines correctly, which
``evaluate.landmark_scores`` says it does well (Uno's C ≈ 0.87–0.89).

``analytic_threshold_penalty`` measures the difference between the two, which
turns a methodological caveat into a number: what trusting the uncalibrated
probability would have cost.

What the baselines are, and why these two
------------------------------------------
The comparison is not against nothing. It is against the two policies operators
actually run:

* **Run-to-failure** — never pull. The right policy when failure is cheap.
* **Fixed-interval** — pull every engine at a set age, swept across ages so the
  comparison is against the *best* version of it, not a strawman. This is the
  incumbent in most maintenance organisations and the honest thing to beat. A
  condition-based policy that cannot beat a well-chosen calendar is not worth
  the sensors.

Ground truth, stated plainly
-----------------------------
Scoring uses each engine's true lifetime from ``evaluation_panel``, which the
models never see. That counterfactual is what makes a policy measurable rather
than merely arguable, and it exists only because this is simulator output. The
README says so.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from src import survival

# Cost of an unplanned in-service failure as a multiple of a planned removal.
# Maintenance-economics studies of aviation and heavy equipment put the ratio
# in the 3-10x range once collateral damage, unscheduled downtime and schedule
# disruption are counted. Five is a mid-range default and nothing rests on it:
# `cost_ratio_sensitivity` sweeps 2-20x, and the recommended threshold is
# reported as a function of the ratio rather than for one assumed value.
DEFAULT_COST_RATIO = 5.0
COST_PULL = 1.0

# Grid of alarm thresholds. Log-spaced because the interesting region is near
# zero — predicted 30-cycle risks are mostly small, so a linear grid would
# spend most of its points where no engine ever sits.
THRESHOLD_GRID = np.unique(np.concatenate([
    np.logspace(-4, -0.3, 60),
    np.linspace(0.5, 0.99, 20),
]))

FIXED_INTERVAL_GRID = np.arange(100, 340, 5)

N_BOOTSTRAP = 400
RANDOM_SEED = 20260801


def score_paths(result, evaluation_panel: pd.DataFrame, feature_columns: list[str],
                edges: np.ndarray, risk_window_start: int,
                horizon: int = survival.DEFAULT_HORIZON) -> pd.DataFrame:
    """Predicted risk at every cycle of every engine's real life.

    Computed once and reused across the whole threshold sweep — the risk path
    does not depend on the threshold, only the decision does, so recomputing it
    per threshold would multiply the work by several hundred for no gain.

    Cycles before the risk window are given zero risk. That is not an
    assumption smuggled in: the model is explicitly conditional on surviving to
    that cycle (see ``survival.restrict_to_risk_window``) and has nothing to say
    before it. No engine fails there either, so the alarm has nothing to catch.
    """
    panel = evaluation_panel.copy()
    in_window = panel["cycle"] >= risk_window_start

    risk = np.zeros(len(panel), dtype=float)
    if in_window.any():
        risk[in_window.to_numpy()] = survival.horizon_risk(
            result, panel.loc[in_window], feature_columns, edges, horizon=horizon
        )
    panel["predicted_risk"] = risk
    return panel


def simulate_threshold(paths: pd.DataFrame, threshold: float,
                       cost_ratio: float = DEFAULT_COST_RATIO,
                       cost_pull: float = COST_PULL) -> dict:
    """Cost of running the fleet with the alarm set at ``threshold``.

    An engine is pulled at the first cycle its predicted risk reaches the
    threshold. If that never happens before its true failure, it fails.
    """
    cost_fail = cost_pull * cost_ratio

    alarms = paths.loc[paths["predicted_risk"] >= threshold]
    first_alarm = alarms.groupby("unit_id")["cycle"].min()

    units = paths.groupby("unit_id")["true_lifetime"].first()
    pull_cycle = first_alarm.reindex(units.index)

    # An alarm at or after the failure cycle is not a save — the engine had
    # already failed. Treating it as one is the single easiest way to
    # accidentally report a policy that beats reality.
    pulled = pull_cycle.notna() & (pull_cycle < units)

    service = np.where(pulled, pull_cycle.fillna(0), units)
    cost = np.where(pulled, cost_pull, cost_fail)
    lead_time = np.where(pulled, units - pull_cycle.fillna(0), np.nan)

    total_service = float(service.sum())
    return {
        "threshold": float(threshold),
        "cost_per_1000_cycles": 1000.0 * float(cost.sum()) / total_service,
        "units": int(len(units)),
        "pulled": int(pulled.sum()),
        "failed": int((~pulled).sum()),
        "failure_rate": float((~pulled).mean()),
        "mean_service_cycles": float(service.mean()),
        "service_utilisation": total_service / float(units.sum()),
        "mean_lead_time": float(np.nanmean(lead_time)) if pulled.any() else np.nan,
        "median_lead_time": float(np.nanmedian(lead_time)) if pulled.any() else np.nan,
    }


def sweep_thresholds(paths: pd.DataFrame,
                     thresholds: np.ndarray = THRESHOLD_GRID,
                     cost_ratio: float = DEFAULT_COST_RATIO) -> pd.DataFrame:
    """Cost across the whole grid, so the optimum is located rather than assumed."""
    return pd.DataFrame([
        simulate_threshold(paths, threshold, cost_ratio=cost_ratio)
        for threshold in thresholds
    ])


def simulate_fixed_interval(paths: pd.DataFrame, interval: int,
                            cost_ratio: float = DEFAULT_COST_RATIO,
                            cost_pull: float = COST_PULL) -> dict:
    """The incumbent policy: pull every engine at a fixed age, sensors ignored.

    Scored on exactly the same cost metric as the model policy, so the
    comparison is like for like. An engine that fails before the interval fails;
    everything else is pulled on schedule.
    """
    cost_fail = cost_pull * cost_ratio
    units = paths.groupby("unit_id")["true_lifetime"].first()

    pulled = units > interval
    service = np.where(pulled, interval, units)
    cost = np.where(pulled, cost_pull, cost_fail)

    return {
        "interval": int(interval),
        "cost_per_1000_cycles": 1000.0 * float(cost.sum()) / float(service.sum()),
        "units": int(len(units)),
        "pulled": int(pulled.sum()),
        "failed": int((~pulled).sum()),
        "failure_rate": float((~pulled).mean()),
        "mean_service_cycles": float(service.mean()),
        "mean_lead_time": float((units[pulled] - interval).mean()) if pulled.any() else np.nan,
    }


def sweep_fixed_interval(paths: pd.DataFrame,
                         intervals: np.ndarray = FIXED_INTERVAL_GRID,
                         cost_ratio: float = DEFAULT_COST_RATIO) -> pd.DataFrame:
    """Sweep the calendar policy, so the model is compared against its best form."""
    return pd.DataFrame([
        simulate_fixed_interval(paths, int(interval), cost_ratio=cost_ratio)
        for interval in intervals
    ])


def run_to_failure(paths: pd.DataFrame,
                   cost_ratio: float = DEFAULT_COST_RATIO,
                   cost_pull: float = COST_PULL) -> dict:
    """Never pull. The floor the other policies have to be worth beating."""
    units = paths.groupby("unit_id")["true_lifetime"].first()
    return {
        "policy": "run-to-failure",
        "cost_per_1000_cycles": 1000.0 * cost_pull * cost_ratio * len(units) / float(units.sum()),
        "units": int(len(units)),
        "failed": int(len(units)),
        "failure_rate": 1.0,
        "mean_service_cycles": float(units.mean()),
    }


def compare_policies(paths: pd.DataFrame,
                     cost_ratio: float = DEFAULT_COST_RATIO) -> dict:
    """The headline table: best model policy against the best of each baseline."""
    model_sweep = sweep_thresholds(paths, cost_ratio=cost_ratio)
    interval_sweep = sweep_fixed_interval(paths, cost_ratio=cost_ratio)

    best_model = model_sweep.loc[model_sweep["cost_per_1000_cycles"].idxmin()]
    best_interval = interval_sweep.loc[interval_sweep["cost_per_1000_cycles"].idxmin()]
    baseline = run_to_failure(paths, cost_ratio=cost_ratio)

    return {
        "cost_ratio": cost_ratio,
        "model": best_model.to_dict(),
        "fixed_interval": best_interval.to_dict(),
        "run_to_failure": baseline,
        "saving_vs_fixed_interval_pct": 100.0 * (
            best_interval["cost_per_1000_cycles"] - best_model["cost_per_1000_cycles"]
        ) / best_interval["cost_per_1000_cycles"],
        "saving_vs_run_to_failure_pct": 100.0 * (
            baseline["cost_per_1000_cycles"] - best_model["cost_per_1000_cycles"]
        ) / baseline["cost_per_1000_cycles"],
        "model_sweep": model_sweep,
        "interval_sweep": interval_sweep,
    }


def analytic_threshold_penalty(paths: pd.DataFrame,
                               cost_ratio: float = DEFAULT_COST_RATIO) -> dict:
    """What trusting the uncalibrated probabilities would have cost.

    The textbook threshold for a calibrated model is ``c_pull / c_fail``, i.e.
    ``1 / cost_ratio``. Because the model's probabilities understate risk, that
    threshold fires too late. This puts a number on the gap instead of leaving
    "the model is not well calibrated" as an unquantified caveat — which is the
    difference between naming a limitation and measuring one.
    """
    analytic = 1.0 / cost_ratio
    at_analytic = simulate_threshold(paths, analytic, cost_ratio=cost_ratio)

    sweep = sweep_thresholds(paths, cost_ratio=cost_ratio)
    best = sweep.loc[sweep["cost_per_1000_cycles"].idxmin()]

    return {
        "analytic_threshold": analytic,
        "analytic_cost": at_analytic["cost_per_1000_cycles"],
        "empirical_threshold": float(best["threshold"]),
        "empirical_cost": float(best["cost_per_1000_cycles"]),
        "penalty_pct": 100.0 * (
            at_analytic["cost_per_1000_cycles"] - best["cost_per_1000_cycles"]
        ) / best["cost_per_1000_cycles"],
        "analytic_failure_rate": at_analytic["failure_rate"],
        "empirical_failure_rate": float(best["failure_rate"]),
    }


def cost_ratio_sensitivity(paths: pd.DataFrame,
                           ratios: tuple[float, ...] = (2, 3, 5, 8, 12, 20)) -> pd.DataFrame:
    """The recommended threshold as a function of the cost ratio.

    The ratio of failure cost to removal cost is the one input a maintenance
    organisation genuinely knows better than this analysis does, and it varies
    by fleet and by contract. Reporting one optimum for one assumed ratio would
    hand over a number that quietly depends on a guess. Reporting the curve
    hands over something usable: read off your own ratio, take the threshold.
    """
    rows: list[dict] = []
    for ratio in ratios:
        comparison = compare_policies(paths, cost_ratio=float(ratio))
        rows.append({
            "cost_ratio": float(ratio),
            "optimal_threshold": comparison["model"]["threshold"],
            "model_cost": comparison["model"]["cost_per_1000_cycles"],
            "model_failure_rate": comparison["model"]["failure_rate"],
            "model_mean_lead_time": comparison["model"]["mean_lead_time"],
            "best_interval": comparison["fixed_interval"]["interval"],
            "interval_cost": comparison["fixed_interval"]["cost_per_1000_cycles"],
            "run_to_failure_cost": comparison["run_to_failure"]["cost_per_1000_cycles"],
            "saving_vs_interval_pct": comparison["saving_vs_fixed_interval_pct"],
        })
    return pd.DataFrame(rows)


def lead_time_distribution(paths: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """How much warning the alarm actually gives, engine by engine.

    The number that decides whether any of this is operationally usable. A
    policy with a superb cost curve and a two-cycle median lead time cannot be
    acted on — you cannot get a shop slot, a part and a crew inside two flights.
    Reporting the distribution rather than the mean matters because the left
    tail is the operational risk: the mean is comfortable precisely when a
    minority of engines give almost no notice.
    """
    alarms = paths.loc[paths["predicted_risk"] >= threshold]
    first_alarm = alarms.groupby("unit_id")["cycle"].min()
    units = paths.groupby("unit_id")["true_lifetime"].first()

    frame = pd.DataFrame({"true_lifetime": units, "alarm_cycle": first_alarm.reindex(units.index)})
    frame["pulled"] = frame["alarm_cycle"].notna() & (frame["alarm_cycle"] < frame["true_lifetime"])
    frame["lead_time"] = np.where(
        frame["pulled"], frame["true_lifetime"] - frame["alarm_cycle"], np.nan
    )
    return frame.reset_index()


def unit_outcome_matrices(paths: pd.DataFrame,
                          thresholds: np.ndarray = THRESHOLD_GRID,
                          intervals: np.ndarray = FIXED_INTERVAL_GRID,
                          cost_ratio: float = DEFAULT_COST_RATIO,
                          cost_pull: float = COST_PULL) -> dict:
    """Per-engine service and cost under every threshold and every interval.

    Precomputation for the bootstrap. Each engine's outcome under a given
    threshold does not depend on which other engines are in the sample, so the
    whole ``engines × thresholds`` grid can be built once and every bootstrap
    replicate becomes two column sums over a resampled row index. That turns a
    ten-minute loop into about a second, and — more importantly — it makes each
    replicate exactly the same computation as the point estimate rather than a
    re-implementation of it that could drift.
    """
    lifetimes = paths.groupby("unit_id")["true_lifetime"].first().sort_index()
    unit_index = {unit: position for position, unit in enumerate(lifetimes.index)}
    n_units = len(lifetimes)
    lives = lifetimes.to_numpy(dtype=float)
    cost_fail = cost_pull * cost_ratio

    # First cycle at which each engine's risk reaches each threshold. Computed
    # by sorting risks descending once: an engine alarms at threshold t exactly
    # when its running maximum risk first reaches t.
    alarm = np.full((n_units, len(thresholds)), np.inf)
    for unit, frame in paths.groupby("unit_id"):
        row = unit_index[unit]
        cycles = frame["cycle"].to_numpy(dtype=float)
        running_max = np.maximum.accumulate(frame["predicted_risk"].to_numpy(dtype=float))
        # searchsorted on the monotone running maximum finds, for each
        # threshold, the first cycle at which it is reached.
        positions = np.searchsorted(running_max, thresholds, side="left")
        reached = positions < len(cycles)
        alarm[row, reached] = cycles[positions[reached]]

    pulled = alarm < lives[:, None]
    service = np.where(pulled, alarm, lives[:, None])
    cost = np.where(pulled, cost_pull, cost_fail)

    interval_grid = np.asarray(intervals, dtype=float)
    interval_pulled = lives[:, None] > interval_grid[None, :]
    interval_service = np.where(interval_pulled, interval_grid[None, :], lives[:, None])
    interval_cost = np.where(interval_pulled, cost_pull, cost_fail)

    return {
        "unit_ids": lifetimes.index.to_numpy(),
        "true_lifetime": lives,
        "thresholds": np.asarray(thresholds, dtype=float),
        "service": service,
        "cost": cost,
        "intervals": interval_grid,
        "interval_service": interval_service,
        "interval_cost": interval_cost,
    }


def bootstrap_optimum(paths: pd.DataFrame, cost_ratio: float = DEFAULT_COST_RATIO,
                      n_boot: int = N_BOOTSTRAP, seed: int = RANDOM_SEED) -> dict:
    """Confidence band on the optimal threshold and its cost, resampling engines.

    A single optimum read off a single sweep is a point estimate with no
    interval, and on a couple of hundred holdout engines the sweep is not
    smooth enough to take its argmin at face value. Resampling **engines** —
    the independent unit, not the row — gives the interval, and its width is
    the honest statement of how precisely the threshold is actually known.
    """
    matrices = unit_outcome_matrices(paths, cost_ratio=cost_ratio)
    service, cost = matrices["service"], matrices["cost"]
    interval_service, interval_cost = matrices["interval_service"], matrices["interval_cost"]
    grid = matrices["thresholds"]
    n_units = len(matrices["true_lifetime"])

    generator = np.random.default_rng(seed)
    thresholds: list[float] = []
    costs: list[float] = []
    savings: list[float] = []

    for _ in range(n_boot):
        index = generator.integers(0, n_units, size=n_units)
        curve = 1000.0 * cost[index].sum(axis=0) / service[index].sum(axis=0)
        best = int(np.argmin(curve))

        interval_curve = (
            1000.0 * interval_cost[index].sum(axis=0) / interval_service[index].sum(axis=0)
        )
        best_interval = float(interval_curve.min())

        thresholds.append(float(grid[best]))
        costs.append(float(curve[best]))
        savings.append(100.0 * (best_interval - curve[best]) / best_interval)

    return {
        "n_bootstrap": n_boot,
        "threshold_median": float(np.median(thresholds)),
        "threshold_ci": (float(np.percentile(thresholds, 2.5)),
                         float(np.percentile(thresholds, 97.5))),
        "cost_median": float(np.median(costs)),
        "cost_ci": (float(np.percentile(costs, 2.5)),
                    float(np.percentile(costs, 97.5))),
        "saving_vs_interval_median": float(np.median(savings)),
        "saving_vs_interval_ci": (float(np.percentile(savings, 2.5)),
                                  float(np.percentile(savings, 97.5))),
    }
'''
# registered before exec: @dataclass resolves its own module during class creation
sys.modules['src.policy'] = _m
setattr(_src, 'policy', _m)
exec(compile(_code_policy, 'src/policy.py', 'exec'), _m.__dict__)

print('inlined modules:', ['acquisition', 'panel', 'survival', 'evaluate', 'figures', 'policy'])

# Turbofan Maintenance Timing

## When should you pull a machine out of service?

**A condition-based pull rule costs 31% less than the best possible fixed-interval
schedule — at exactly the same failure rate.** Pull a unit when its predicted 30-cycle
failure risk crosses 7.7%, and you get the same 0.9% in-service failure rate as a
140-cycle calendar schedule while delivering 204 cycles of service per unit instead of
140. Same safety, 46% more working life, with a median 25 cycles of warning.

Getting there required fixing something the standard treatment of this problem gets
wrong. Almost every published analysis of engine degradation data assumes you can watch
every machine fail. You cannot — at any moment a real fleet is *mostly machines that have
not failed yet*, and those are exactly the machines a maintenance decision is about.
Dropping them, which is what "regress on remaining useful life" is forced to do, is not
just wasteful: it is **biased**, understating average life by 6.3% here and by more as the
fleet gets more realistic.

This notebook censors a complete run-to-failure benchmark **on purpose**, measures that
bias against a truth that is actually known, builds the censoring-aware alternative, and
converts it into a maintenance threshold with a confidence interval.

---

*Data: NASA C-MAPSS turbofan degradation benchmark (public domain). It is simulator
output, not a real fleet, and is used here as a stated analogue for industrial
maintenance-timing problems. No employer's data is involved.*

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import Image, display

from src import acquisition, evaluate, figures, panel, policy, survival

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

### Loading the data

The loader below works in three situations without changing a line: a repo checkout where
`src/acquisition.py` has already run, a Kaggle notebook with the benchmark attached as a
data source, and a fresh checkout where nothing has been downloaded yet. In every case the
parsing goes through the **same** `src/acquisition` functions — the schema checks that
reject a file with cycle gaps or missing units apply identically wherever the bytes came
from.

In [ ]:
def load_trajectories() -> pd.DataFrame:
    # Read the training trajectories, whatever environment this is running in.
    if acquisition.TRAIN_PARQUET.exists():
        print(f"reading {acquisition.TRAIN_PARQUET.name} from the repo's data/raw")
        return pd.read_parquet(acquisition.TRAIN_PARQUET)

    # On Kaggle the benchmark arrives as the raw .txt files under /kaggle/input.
    # Searched for rather than passed in, so this cell needs no configuration and
    # the notebook is identical in both environments.
    for root in (Path("/kaggle/input"), acquisition.EXTRACT_DIR):
        if not root.exists():
            continue
        found = sorted(root.rglob("train_FD00*.txt"))
        if found:
            print(f"parsing {len(found)} trajectory files from {root}")
            return pd.concat(
                [acquisition.read_trajectory_file(path, path.stem.split("_")[1])
                 for path in found],
                ignore_index=True,
            )

    print("no local copy found - downloading from NASA")
    return acquisition.main()


trajectories = load_trajectories()
engines = trajectories.groupby(["subset", "unit"]).ngroups
print(f"\n{len(trajectories):,} unit-cycles from {engines} engines")
trajectories.head(3)

## 1. The decision this answers

At each cycle an operator can leave a machine in service or pull it for planned
maintenance:

- **Pull early** — cheap, but you throw away remaining useful life.
- **Leave it in** — you risk an in-service failure, which costs several times a planned
  removal once collateral damage, unscheduled downtime and schedule disruption are counted.

So the policy is a single number: an alarm threshold on predicted risk. The question is
where to put it, and everything before the final section exists to earn the right to
answer that.

Three sub-questions structure the work:

1. **What does ignoring censoring actually cost?** Not "is it bad in principle" — how many
   cycles, and does it get worse or better as the fleet becomes more realistic?
2. **Which model, and can it be justified rather than asserted?** The reflex choice is Cox
   proportional hazards. Its central assumption is testable, so it gets tested.
3. **Where should the alarm go, and how precisely is that known?** A threshold without an
   interval is a guess with a decimal point.

## 2. The data

**NASA C-MAPSS Turbofan Engine Degradation Simulation** (Saxena & Goebel, 2008), from the
NASA Prognostics Data Repository. Work of the U.S. Government — public domain.

NASA ran an engine simulator to failure on fleets of engines, each starting with a
different unstated amount of initial wear. Every row is one engine on one flight cycle: 3
operational settings and 21 sensor channels. Four subsets vary the number of operating
conditions and the number of fault modes.

`src/acquisition.py` tries three known mirrors in order, records which one served,
checksums the archive and every extracted file into `data/raw/provenance.json`, and
asserts the published unit counts — so a repackaged mirror fails loudly rather than
quietly shrinking the fleet.

**Why this dataset, honestly stated.** It is simulator output, so nothing here is a claim
about real turbofans. It is used because it has the one property the argument needs and
almost no public maintenance dataset has: **complete run-to-failure trajectories for every
unit**. That is what makes it possible to remove information on purpose and then check the
answer against a truth that is genuinely known.

In [ ]:
summary = (
    trajectories.groupby("subset")
    .agg(engines=("unit", "nunique"), rows=("cycle", "size"),
         shortest_life=("cycle", "max"))
    .assign(**{
        "median life": trajectories.groupby(["subset", "unit"])["cycle"].max()
                                   .groupby("subset").median().astype(int),
        "shortest life": trajectories.groupby(["subset", "unit"])["cycle"].max()
                                     .groupby("subset").min(),
        "longest life": trajectories.groupby(["subset", "unit"])["cycle"].max()
                                    .groupby("subset").max(),
    })
    .drop(columns="shortest_life")
)
for subset, design in acquisition.SUBSETS.items():
    summary.loc[subset, "conditions"] = design["conditions"]
    summary.loc[subset, "fault modes"] = design["faults"]
summary

## 3. What the data looks like, and what that forces

Only the observations that changed a modelling decision are shown. Each one below made
something in `src/` be written the way it is.

### 3.1 The operating condition dominates the raw sensors

In the six-condition subsets, raw sensor variance is mostly about *which condition the
engine is flying*, not how worn it is. Sensor 7's raw standard deviation is 0.9 in FD001
and 146 in FD002 — that hundredfold difference is altitude and throttle.

Feed those raw values to a model pooling the subsets and it will learn the flight profile
and call it wear.

In [ ]:
sensors = [f"sensor_{i}" for i in range(1, 22)]
raw_spread = trajectories.groupby("subset")[["sensor_7", "sensor_9", "sensor_12"]].std()
print("Raw sensor standard deviation, by subset:")
display(raw_spread.round(2))

trajectories["regime"] = panel.assign_regime(trajectories)
print(f"\nOperating regimes recovered by rounding the settings: {trajectories['regime'].nunique()}")
print(trajectories.groupby("subset")["regime"].nunique().to_string())

**Takeaway:** six regimes, recovered by *rounding* the three operational settings rather
than by clustering. The settings fall into six tight, well-separated groups, so rounding
recovers them exactly — and unlike k-means it is deterministic, needs no seed, and can be
read in one line.

One trap worth naming: FD001 stores its setting as `-0.0` where FD002 stores `0.0`.
Formatted naively that splits one physical condition into two and produces a phantom
seventh regime. `panel.assign_regime` normalises it, and a test pins the behaviour.

Every sensor is then standardised **within its regime**, so a reading means "unusual for
this operating point".

In [ ]:
frame = trajectories.copy()
kept, dropped = panel.screen_sensors(frame)
print(f"sensors kept:    {len(kept)}")
print(f"sensors dropped: {dropped}")
print("\nThese four are exactly constant within every regime - they carry no information")
print("and would make the design matrix singular. Screened by a variance rule applied to")
print("the data, not by a hard-coded list.")

### 3.2 Every engine wears differently — which is the case for a per-unit model

The left panel is the argument for reading each unit's own trajectory rather than the
calendar. The right panel shows the same signal aligned on failure, which makes the
degradation obvious — and which an operator can never do, because that axis is only
knowable in hindsight.

In [ ]:
prepared = panel.prepare(trajectories)
risk_set = prepared["risk_set"]
fleet = prepared["fleet"]
features = prepared["feature_columns"]

print(f"regimes            : {prepared['n_regimes']}")
print(f"sensors kept       : {len(prepared['sensors_kept'])}")
print(f"censored fraction  : {prepared['censoring']['realised_rate']:.1%} "
      f"(target {prepared['censoring']['target_rate']:.0%})")
print(f"horizon ceiling U  : {prepared['censoring']['horizon_ceiling']} cycles")
print(f"risk-set rows      : {len(risk_set):,}")

Image(filename=str(figures.degradation_trajectories(risk_set)))

## 4. Method

### 4.1 Removing information on purpose

This is the decision the whole project rests on, so it is worth stating precisely.

Each unit *i* has a true failure cycle `T_i`, known from the raw data and hidden from every
model. It is assigned a snapshot horizon `C_i` drawn uniformly from `{1..U}`,
**independently of `T_i`**. The unit is then observed for `Y_i = min(T_i, C_i)` cycles,
with event indicator `δ_i = 1{T_i ≤ C_i}`.

Three properties this buys, each of which matters:

- **Independent censoring holds by construction.** Kaplan-Meier and the partial likelihood
  both require it; here it is true by design rather than hoped for.
- **It is administrative, not left-truncating.** Each unit is watched from cycle 1, so
  there is no delayed entry to handle at the unit level.
- **It is reproducible.** `U` is *solved for* by bisection to hit a target censored
  fraction, so the censoring rate is a chosen property of the design rather than whatever
  fell out of an arbitrary constant.

Implemented in `panel.induce_censoring` and `panel.calibrate_horizon`.

In [ ]:
lifetimes = panel.unit_lifetimes(trajectories)
for target in (0.2, 0.4, 0.6):
    ceiling = panel.calibrate_horizon(lifetimes["true_lifetime"].to_numpy(), target)
    print(f"target {target:.0%} censored -> horizon ceiling U = {ceiling:>4} cycles")

correlation = np.corrcoef(fleet["horizon"], fleet["true_lifetime"])[0, 1]
print(f"\ncorrelation(horizon, true lifetime) = {correlation:+.4f}")
print("Near zero, as it must be: the censoring must not see what it is censoring.")

### 4.2 Two panels, and why the separation is load-bearing

- **`risk_set`** — truncated at the snapshot. The **only** thing any model is fitted on.
- **`evaluation_panel`** — the complete run to failure. Used **only** to simulate what a
  policy would have cost.

A maintenance policy has to be scored on what *would have happened*: if the alarm fires at
cycle 190 and the engine would have run to 240, the policy bought 50 cycles of avoided
failure and gave up 50 cycles of service. That counterfactual needs the whole trajectory,
including the part the models are forbidden to see.

Being able to build the second panel at all is the reason a simulator benchmark is worth
using. On a real fleet the counterfactual is unobservable and a policy can only be argued
for, never measured.

### 4.3 Drift, not level

Engines enter service with different, unstated initial wear, so a sensor's *level*
confounds "born worn" with "worn out". The change since the unit's own first few cycles
separates them.

That is a claim about out-of-sample behaviour, so it is tested out of sample rather than
asserted — see section 5.2.

### 4.4 A ladder of models

Each rung exists to justify the next.

In [ ]:
# Rung 1: no covariates, no functional form. Do the fleet designs differ at all?
curves = survival.kaplan_meier(fleet)
logrank = survival.logrank(fleet)
print(f"log-rank across the four subsets: chi2 = {logrank['test_statistic']:.1f}, "
      f"df = {logrank['degrees_of_freedom']}, p = {logrank['p_value']:.2e}")
Image(filename=str(figures.survival_curves(curves)))

They differ, decisively. Only now is it legitimate for a model to condition on subset.

Next, two decisions the data forced.

In [ ]:
fit_all = risk_set.loc[risk_set["split"] == "fit"]
window_start = survival.risk_window_start(fit_all)

fit = survival.restrict_to_risk_window(fit_all, window_start)
holdout = survival.restrict_to_risk_window(
    risk_set.loc[risk_set["split"] == "holdout"], window_start
)
edges = survival.derive_band_edges(fit)

print(f"No unit in this fleet fails before cycle {window_start}.")
print(f"  rows below it : {len(fit_all) - len(fit):,}  (constant outcome, zero failures)")
print(f"  engines lost  : {fit_all['unit_id'].nunique() - fit['unit_id'].nunique()}"
      f" of {fit_all['unit_id'].nunique()} (censored before the window opens)")
print(f"\nhazard band edges: {edges.astype(int).tolist()}")

bands = survival.assign_band(fit["cycle"].to_numpy(), edges)
print("failures per band:",
      pd.Series(fit["failed"].to_numpy()).groupby(bands).sum().to_dict())

**Two things just happened, and both are modelling decisions rather than housekeeping.**

**Left truncation at cycle 128.** Those ~40,000 early rows have a *constant* outcome — no
failure ever occurs there — so a logistic model asked to explain them drives its intercept
to minus infinity. That is textbook complete separation: the fit does not converge and the
Hessian is singular. Restricting the risk set is the honest response, and it has a name:
the hazard is estimated **conditional on surviving to cycle 128**. The model says nothing
about the first 127 cycles, and it should not — there is no failure there to learn from.
The price is 106 engines censored before the window opens, which drop out entirely.

**Equal-event hazard bands.** Equal-*width* bands fail in the sparse tail, where several
20-cycle bands hold a few dozen rows and no failures at all — again a minus-infinity
coefficient. Cutting at the quantiles of observed failure cycles gives every band roughly
thirty events and none empty. The edges are derived from the **fit set only**; quantiles of
holdout failure times would be the holdout's answers leaking into the design matrix.

In [ ]:
# Rung 2: Cox proportional hazards, fitted so its assumption can be tested.
cox = survival.fit_cox_time_varying(fit, features)
ph = survival.test_proportional_hazards(cox, fit, features)

print(f"Proportional hazards violated by {int(ph['violates_ph'].sum())} of {len(features)} "
      f"covariates (Holm-corrected for multiple testing)\n")
ph.head(8).round(4)

Cox's central assumption — that a covariate's effect on the log-hazard is the same at cycle
130 as at cycle 350 — **fails on 10 of 17 covariates**. Worse for Cox, the violations are
concentrated in the *highest-effect* sensors (4, 11, 2, 3, 17): their influence weakens as
a unit ages.

A note on how that was tested, because the obvious version of the test is wrong here. The
shortcut — correlate raw covariate values at failures with failure time — tests something
else entirely: the drift features **rise with age by construction**, so it flags
essentially every covariate whether or not the model's assumption is broken.
`survival.schoenfeld_residuals` subtracts the model's own expectation at each failure time,
which is exactly what removes that artefact.

So the workhorse is the discrete-time model, for three reasons in order of weight:

1. **The decision needs an absolute probability, not a ranking.** Cox's partial likelihood
   deliberately factors the baseline hazard out. "Pull when 30-cycle risk exceeds *p*" is a
   statement about an absolute probability.
2. **Time is discrete.** One inspection per cycle, fail or not — a Bernoulli trial per
   cycle at risk, which is what a logistic regression on the risk set estimates. Cox assumes
   continuous time in which exact ties cannot occur; here ties *are* the data.
3. **Duration dependence is the question**, and PH has just been shown to fail.

In [ ]:
# Rung 3: the workhorse. Standard errors clustered by unit - one engine contributes
# hundreds of correlated rows, and treating them as independent would shrink the
# standard errors by roughly sqrt(average unit length).
hazard = survival.fit_discrete_hazard(fit, features, edges)
coefficients = survival.tidy(hazard)

print(f"parameters : {len(hazard.params)}")
print(f"events     : {int(fit['failed'].sum())} over {len(fit):,} unit-cycles at risk")
print(f"converged  : {hazard.mle_retvals['converged']}")
print(f"pseudo-R2  : {hazard.prsquared:.3f}\n")

sensor_terms = coefficients[~coefficients["term"].str.startswith(("band_", "const"))]
sensor_terms.sort_values("p_value").head(6).round(4)

In [ ]:
Image(filename=str(figures.baseline_hazard(coefficients, edges)))

The baseline hazard was **estimated, not assumed** — that is the whole reason cycle enters
as band dummies rather than as a linear term. Whether risk accelerates with age is the
question a maintenance policy turns on, so imposing the answer would have been circular.

## 5. Results

### 5.1 What ignoring censoring costs

The measurement the project is built around. Three estimates of the same quantity, against
a truth that is actually known.

In [ ]:
bias = evaluate.censoring_bias(fleet)

comparison = pd.DataFrame([
    {"estimate": "Truth (full trajectories)", "mean life": bias["true_mean_life"],
     "error": 0.0, "error %": 0.0},
    {"estimate": "Complete-case (drop unfailed)", "mean life": bias["complete_case_mean_life"],
     "error": bias["complete_case_bias_cycles"], "error %": bias["complete_case_bias_pct"]},
    {"estimate": "Naive (snapshot as lifetime)", "mean life": bias["naive_observed_mean_life"],
     "error": bias["naive_bias_cycles"],
     "error %": 100 * bias["naive_bias_cycles"] / bias["true_mean_life"]},
]).round(2)
display(comparison)

print(f"\nKaplan-Meier median {bias['kaplan_meier_median']:.0f} vs "
      f"true median {bias['true_median_life']:.0f} cycles "
      f"-> error {bias['kaplan_meier_median_error']:+.0f}")

The direction is not an accident and it is worth being able to say out loud: **a unit is
censored exactly when its lifetime exceeds its horizon**, so deleting censored units
deletes the long-lived ones preferentially. The surviving sample is biased short and the
fleet looks more fragile than it is.

A single censoring rate could always be dismissed as contrived, so it is swept.

In [ ]:
bias_sweep = evaluate.bias_by_censoring_rate(trajectories)
display(bias_sweep.round(2))
Image(filename=str(figures.censoring_bias(bias_sweep)))

The bias grows monotonically with how much of the fleet is still running — which is the
direction that matters, because real fleets are mostly survivors.

The second series is the positive half of the argument, and it is why this project has a
recommendation rather than just a complaint: **the information is not destroyed by
censoring, only by the wrong estimator.** Kaplan-Meier's error stays on zero throughout.

### 5.2 Testing the "drift, not level" claim

In [ ]:
feature_comparison = evaluate.compare_feature_sets(
    fit, holdout, features, prepared["level_columns"], edges, landmarks=(150, 175, 200)
)
feature_comparison.round(4)

The level features fit **much** better in sample (pseudo-R² 0.59 vs 0.40) and discriminate
**worse** out of it (Uno's C 0.83 vs 0.88). That is what overfitting looks like, and it is
why the model uses drift alone. The a-priori argument was right, but it is reported as a
measurement rather than an assertion.

### 5.3 Does it discriminate?

Scoring is done at **landmarks**: fix a cycle, keep the units still running, use their
readings at that instant, and score how well predicted risk orders what happens next. A
time-varying-covariate model does not have one C-index; it has one per moment at which you
might ask.

In [ ]:
scores = evaluate.landmark_scores(hazard, holdout, features, edges, fit,
                                  landmarks=(150, 175, 200))
display(scores.round(4))

brier = evaluate.brier(hazard, holdout, features, edges, fit, 150)
print(f"\nintegrated Brier score      : {brier['integrated_brier_score']:.4f}")
print(f"Kaplan-Meier baseline       : {brier['baseline_integrated_brier_score']:.4f}")
print(f"improvement                 : {brier['improvement']:.4f}")

Both concordance indices are reported deliberately. **Harrell's C** is biased under
censoring — the comparable pairs it can form are not a random sample. **Uno's C**
reweights by the inverse probability of remaining uncensored and is not. In a project whose
subject is censoring, quoting only the censoring-sensitive metric would be self-defeating.

The Brier comparison is the reference every prognostic model should have to clear: the
fleet-wide Kaplan-Meier curve uses no sensor information at all. Beating it is evidence
the sensors are being read to some effect.

### 5.4 The weakness, measured rather than mentioned

In [ ]:
calibration = evaluate.calibration(hazard, holdout, features, edges, 150)
display(calibration.round(4))
Image(filename=str(figures.calibration(calibration, 150)))

**The model is badly under-confident in the top deciles** — it predicts 62% where the
observed rate is 100%. The mechanism is known and is a direct consequence of a design
choice: `survival.horizon_risk` compounds the per-cycle hazard forward while **freezing
the sensor readings at their current value**, so a unit degrading unusually fast has its
future understated.

That freezing is not laziness. At the moment of the decision, today's readings are all the
operator has; a model that needed tomorrow's would be unusable. But it has a consequence
for the decision layer, and section 6 is built around it rather than around hoping it does
not matter.

## 6. The decision

### 6.1 The cost metric, and the trap in the obvious one

Score a policy on cost **per unit** and the optimum is degenerate: pull every machine on
cycle 1, nothing ever fails, every removal is the cheap kind, and the fleet does no work.

The metric has to charge for the service given up:

$$\text{cost per 1,000 cycles} = \frac{1000 \times \sum_i \text{cost}_i}{\sum_i \text{cycles delivered}_i}$$

Pulling early buys a lower numerator at the price of a smaller denominator. That is the
real trade, and it cannot be gamed by being conservative.

### 6.2 Why the threshold is swept, not calculated

For a **calibrated** model the answer is analytic: act when risk exceeds `c_pull / c_fail`.
Section 5.4 showed this model is not calibrated, so that formula fires too late.

Two responses were available. Recalibrate — but the outcome needed to fit a recalibration
is itself censored, which repairs one bias by introducing another. Or stop relying on the
probabilities being right: **sweep** the threshold and pick the one that actually costs
least. That makes the threshold a *rank* cut, and the model only has to order units
correctly — which it does, at Uno's C ≈ 0.88.

The gap between the two is measured below rather than left as a caveat.

In [ ]:
evaluation_panel = prepared["evaluation_panel"]
paths = policy.score_paths(
    hazard, evaluation_panel.loc[evaluation_panel["split"] == "holdout"],
    features, edges, window_start
)

comparison = policy.compare_policies(paths)
bootstrap = policy.bootstrap_optimum(paths)

model, interval = comparison["model"], comparison["fixed_interval"]
table = pd.DataFrame([
    {"policy": "Run to failure",
     "cost per 1,000 cycles": comparison["run_to_failure"]["cost_per_1000_cycles"],
     "failure rate": comparison["run_to_failure"]["failure_rate"],
     "service per unit": comparison["run_to_failure"]["mean_service_cycles"]},
    {"policy": f"Best fixed interval ({interval['interval']:.0f} cycles)",
     "cost per 1,000 cycles": interval["cost_per_1000_cycles"],
     "failure rate": interval["failure_rate"],
     "service per unit": interval["mean_service_cycles"]},
    {"policy": f"Model policy (threshold {model['threshold']:.3f})",
     "cost per 1,000 cycles": model["cost_per_1000_cycles"],
     "failure rate": model["failure_rate"],
     "service per unit": model["mean_service_cycles"]},
]).round(3)
display(table)

print(f"\nSaving vs best fixed interval: {comparison['saving_vs_fixed_interval_pct']:.1f}% "
      f"(95% CI {bootstrap['saving_vs_interval_ci'][0]:.1f}-"
      f"{bootstrap['saving_vs_interval_ci'][1]:.1f}%)")
print(f"Optimal threshold: {model['threshold']:.3f} "
      f"(95% CI {bootstrap['threshold_ci'][0]:.3f}-{bootstrap['threshold_ci'][1]:.3f})")

**The headline, in the clearest form available:** the calendar policy and the model policy
accept **the same 0.9% failure rate**. The calendar gets there by pulling every unit at 140
cycles. The model gets there by pulling each unit when *that unit* looks unwell — at 204
cycles on average. Same safety, 46% more working life.

The baseline is the *best* fixed interval, swept across the whole grid, not a strawman. A
condition-based policy that could not beat a well-chosen calendar would not be worth the
sensors.

In [ ]:
Image(filename=str(figures.threshold_sweep(
    comparison["model_sweep"], model, bootstrap,
    interval["cost_per_1000_cycles"],
    comparison["run_to_failure"]["cost_per_1000_cycles"])))

In [ ]:
Image(filename=str(figures.policy_frontier(comparison["model_sweep"], comparison["interval_sweep"], model)))

### 6.3 What trusting the uncalibrated probability would have cost

In [ ]:
analytic = policy.analytic_threshold_penalty(paths)
pd.DataFrame([
    {"threshold rule": "Textbook (c_pull / c_fail)", "threshold": analytic["analytic_threshold"],
     "cost per 1,000": analytic["analytic_cost"], "failure rate": analytic["analytic_failure_rate"]},
    {"threshold rule": "Swept (chosen by measured cost)", "threshold": analytic["empirical_threshold"],
     "cost per 1,000": analytic["empirical_cost"], "failure rate": analytic["empirical_failure_rate"]},
]).round(4)

The textbook rule costs **21.9% more** and accepts an **8.0% failure rate instead of
0.9%** — an eightfold increase in in-service failures. That is the difference between
naming a limitation and measuring one.

### 6.4 Is it operationally usable?

A policy with a superb cost curve and a two-cycle lead time cannot be acted on — you cannot
get a shop slot, a part and a crew inside two flights. The distribution matters more than
the mean, because the left tail is the operational risk.

In [ ]:
leads = policy.lead_time_distribution(paths, model["threshold"])
pulled = leads.loc[leads["pulled"], "lead_time"]
print(f"engines pulled in time : {int(leads['pulled'].sum())} of {len(leads)}")
print(f"median lead time       : {pulled.median():.0f} cycles")
print(f"10th percentile        : {np.percentile(pulled, 10):.0f} cycles")
print(f"minimum                : {pulled.min():.0f} cycles")

Image(filename=str(figures.lead_times(leads, model["threshold"])))

### 6.5 The one input this analysis does not know

The ratio of failure cost to planned-removal cost varies by fleet and by contract, and a
maintenance organisation knows it better than this analysis does. Reporting a single optimum
for one assumed ratio would hand over a number that quietly depends on a guess. Reporting
the curve hands over something usable.

In [ ]:
sensitivity = policy.cost_ratio_sensitivity(paths)
display(sensitivity.round(3))
Image(filename=str(figures.cost_ratio_sensitivity(sensitivity)))

Across a ten-fold range of cost ratios the saving stays between 28% and 31%, and the
recommended threshold barely moves. The recommendation does not depend on getting the cost
ratio right.

## 7. Limitations

- **The data is simulated.** C-MAPSS is physics-model output with injected noise, not
  measurements from a real fleet. Nothing here is a claim about real turbofans. The
  *methods* transfer; the specific numbers do not.
- **Censoring is induced by design, not observed.** The mechanism is administrative and
  independent of lifetime by construction. Real maintenance data often has **informative**
  censoring — units pulled *because* they looked unwell — which breaks both Kaplan-Meier
  and the partial likelihood. That case is out of scope here and would need a different
  approach: either a joint model of the pull decision and the failure process, or an
  instrument for why a unit was pulled.
- **The model is not well calibrated.** It understates risk in the top deciles, for the
  reason given in 5.4. The decision layer is designed around this rather than pretending
  otherwise, and the cost of ignoring it is quantified at 21.9%.
- **Forward risk freezes the covariates.** A unit degrading unusually fast has its future
  understated, which makes the reported policy gain conservative rather than flattering.
- **Costs are relative, not currency.** A planned removal is 1 and a failure is the
  multiple; no monetary figure is claimed. The sensitivity sweep is what makes the
  recommendation usable without one.
- **The analysis says nothing before cycle 128**, by construction, and 106 fit-set units
  censored before that window drop out of the model fit entirely.

## 8. What I would do about it

**Set the alarm at a predicted 30-cycle failure risk of 0.077**, having first read the
threshold off the sensitivity curve in 6.5 using the organisation's own failure-to-removal
cost ratio. Expect roughly a 31% reduction in cost per cycle of service against a
well-tuned calendar schedule, at the same in-service failure rate, with a median 25 cycles
of warning and a 10th percentile of 12.

**Two conditions on that recommendation**, both of which are checkable before anything is
deployed:

1. **Do not read the probabilities as probabilities.** The threshold is a rank cut chosen
   by measured cost. If someone wants a calibrated risk number for a different purpose, the
   model needs recalibrating first, and doing that properly on censored outcomes is its own
   piece of work.
2. **Verify that pulls are not already condition-driven.** The entire analysis assumes
   censoring is independent of health. In an organisation where units are already pulled
   because they looked unwell, that assumption is false and these estimates are an upper
   bound on how well this works.

**The transferable finding**, which outlives the specific numbers: if you are modelling
time to failure and your method needs a completed lifetime as its target, you are training
on a biased sample of your fleet — and the bias grows with the share of machines still
running, which is to say with how normal your fleet is.

## 9. Reproducing this

From a clean checkout:

```bash
py -3.10 -m venv .venv && .venv/Scripts/pip install -r requirements.txt
python src/acquisition.py
python -m pytest -q
python run_analysis.py
```

`run_analysis.py` regenerates every number in this notebook and in the README, plus all ten
figures, in about 20 seconds. This notebook imports `src/` and calls the same functions the
script does — it never re-implements an analysis path, which is the only way to guarantee
the notebook and the script cannot disagree.

49 tests pin the claims, including a seeded-reproducibility check on the censoring
mechanism and a cost-sweep optimum verified against a fixture whose answer is a calculation
rather than a re-run of the code being tested.

**Data:** Saxena, A. and Goebel, K. (2008). *Turbofan Engine Degradation Simulation Data
Set*. NASA Prognostics Data Repository, NASA Ames Research Center. Work of the U.S.
Government — public domain.

In [ ]:
print("Figures written to outputs/figures/:")
for path in sorted(figures.FIGURE_DIR.glob("*.png")):
    print(f"  {path.name}")